In [1]:
# %%

import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
import numpy as np
import re
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, Optional, List
import fiona
import requests
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from shapely.geometry import Point



# sets the view dimensions
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 80)



/Users/mayarabin/Desktop/Thesis Files/.venv/lib/python3.8/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:


# %%
# downloading of all the census district shapefiles
# later allows us to group cities into split or unified
# from UCLA's standardized files see here: https://cdmaps.polisci.ucla.edu/

# census district shapes for the year 1955 or the districts from 1950 to 1960
path_1950_1960_Dist = "/Users/mayarabin/Desktop/Thesis Downloads V2/Census District Shapefiles 1950-2025/districts084_1955/districts084.shp"

# census district shapes for the year 1965 or the districts from 1960 to 1970
path_1960_1970_Dist = "/Users/mayarabin/Desktop/Thesis Downloads V2/Census District Shapefiles 1950-2025/districts089_1965/districts089.shp"

# census district shapes for the year 1975 or the districts from 1970 to 1980
path_1970_1980_Dist = "/Users/mayarabin/Desktop/Thesis Downloads V2/Census District Shapefiles 1950-2025/districts094_1975/districts094.shp"

# census district shapes for the year 1985 or the districts from 1980 to 1990
path_1980_1990_Dist = "//Users/mayarabin/Desktop/Thesis Downloads V2/Census District Shapefiles 1950-2025/districts099_1985/districts099.shp"

# census district shapes for the year 1995 or the districts from 1990 to 2000
path_1990_2000_Dist = "/Users/mayarabin/Desktop/Thesis Downloads V2/Census District Shapefiles 1950-2025/districts104_1995/districts104.shp"

# census district shapes for the year 2005 or the districts from 2000 to 2010
path_2000_2010_Dist = "/Users/mayarabin/Desktop/Thesis Downloads V2/Census District Shapefiles 1950-2025/districts109_2005/districts109.shp"

# census district shapes for the year 2015 or the districts from 2010 to 2020
path_2010_2020_Dist = "/Users/mayarabin/Desktop/Thesis Downloads V2/Census District Shapefiles 1950-2025/districts114_2015/districts114.shp"

# census district shapes for the year 2025 or the districts from 2020 to 2030
path_2020_Present_Dist = "/Users/mayarabin/Desktop/Thesis Downloads V2/Census District Shapefiles 1950-2025/districts119_2025/districts119.shp"

# %%
# actually downloads in all of the shapefiles for Census Districts 1950 to 2025

gdf_1950s_Dist = gpd.read_file(path_1950_1960_Dist)
gdf_1960s_Dist = gpd.read_file(path_1960_1970_Dist)
gdf_1970s_Dist = gpd.read_file(path_1970_1980_Dist)
gdf_1980s_Dist = gpd.read_file(path_1980_1990_Dist)
gdf_1990s_Dist = gpd.read_file(path_1990_2000_Dist)
gdf_2000s_Dist = gpd.read_file(path_2000_2010_Dist)
gdf_2010s_Dist = gpd.read_file(path_2010_2020_Dist)
gdf_2020s_Dist = gpd.read_file(path_2020_Present_Dist)


# %%
# adding a column to track the year of the change

gdf_1950s_Dist["split_year"] = 1950
gdf_1960s_Dist["split_year"] = 1960
gdf_1970s_Dist["split_year"] = 1970
gdf_1980s_Dist["split_year"] = 1980
gdf_1990s_Dist["split_year"] = 1990
gdf_2000s_Dist["split_year"] = 2000
gdf_2010s_Dist["split_year"] = 2010
gdf_2020s_Dist["split_year"] = 2020




In [3]:
# %%
# combines all of the data on census districts from 1950 to 2020

dfs = [gdf_1950s_Dist, gdf_1960s_Dist, gdf_1970s_Dist, gdf_1980s_Dist,
       gdf_1990s_Dist, gdf_2000s_Dist, gdf_2010s_Dist, gdf_2020s_Dist]
Districts = pd.concat(dfs, ignore_index=True)

gdf_1950s_Dist = gdf_1950s_Dist.to_crs(5070)
gdf_1960s_Dist = gdf_1960s_Dist.to_crs(5070)
gdf_1970s_Dist = gdf_1970s_Dist.to_crs(5070)
gdf_1980s_Dist = gdf_1980s_Dist.to_crs(5070)
gdf_1990s_Dist = gdf_1990s_Dist.to_crs(5070)
gdf_2000s_Dist = gdf_2000s_Dist.to_crs(5070)
gdf_2010s_Dist = gdf_2010s_Dist.to_crs(5070)
gdf_2020s_Dist = gdf_2020s_Dist.to_crs(5070)


# %%
# ============================================================
# COOK PVI LOAD IN
# ============================================================

COOK_DIR = "/Users/mayarabin/Desktop/Thesis Downloads V2/Political Data"

COOK_FILES = {
    1997: "Cook_Index_Congressional_Districts - 1997.csv",
    1999: "Cook_Index_Congressional_Districts - 1999.csv",
    2001: "Cook_Index_Congressional_Districts - 2001.csv",
    2003: "Cook_Index_Congressional_Districts - 2003.csv",
    2005: "Cook_Index_Congressional_Districts - 2005.csv",
    2007: "Cook_Index_Congressional_Districts - 2007.csv",
    2009: "Cook_Index_Congressional_Districts - 2009.csv",
    2011: "Cook_Index_Congressional_Districts - 2011.csv",
    2013: "Cook_Index_Congressional_Districts - 2013.csv",
    2015: "Cook_Index_Congressional_Districts - 2015.csv",
    2017: "Cook_Index_Congressional_Districts - 2017.csv",
    2019: "Cook_Index_Congressional_Districts - 2019.csv",
    2021: "Cook_Index_Congressional_Districts - 2021.csv",
    2023: "Cook_Index_Congressional_Districts - 2023.csv",
    2025: "Cook_Index_Congressional_Districts - 2025.csv",
}



In [4]:
# %%
def find_dist_col(columns: list) -> str:
    """Find whichever column holds the district code (AL-01 style)."""
    for c in columns:
        c_lower = c.strip().lower()
        if c_lower in ("dist", "dist2025", "district", "dist_code"):
            return c
        if re.search(r"dist", c_lower):
            return c
    raise ValueError(f"Could not find district column in: {columns}")


def find_pvi_col(columns: list, year: int) -> str:
    """
    Find the integer PVI column (e.g. 'R+7', not 'R+7.26').
    Avoid raw/decimal columns.
    """
    cols_lower = {c.strip().lower(): c for c in columns}

    candidates = [
        f"pvi{year}",
        f"{year} pvi",
        "pvi",
    ]
    for cand in candidates:
        if cand in cols_lower:
            return cols_lower[cand]

    for c_lower, c_orig in cols_lower.items():
        if "pvi" in c_lower and "raw" not in c_lower and "rank" not in c_lower and "sortable" not in c_lower:
            return c_orig

    raise ValueError(f"Could not find PVI column in: {columns}")



In [5]:
def standardize_dist_code(code: str) -> str:
    """
    Normalize district codes:
      AL-01  →  AL-01   (already good)
      AL-1   →  AL-01   (zero-pad)
      AK-AL  →  AK-01   (at-large → 01)
      AK-00  →  AK-01   (some sources use 00 for at-large)
    """
    code = str(code).strip().upper()
    if not code or code in ("NAN", ""):
        return pd.NA

    if "-" not in code:
        return pd.NA

    state, dist = code.split("-", 1)
    dist = dist.strip()

    if dist in ("AL", "00"):
        dist = "01"

    if dist.isdigit():
        dist = dist.zfill(2)

    return f"{state}-{dist}"


def parse_pvi(pvi_str: str):
    """
    Parse PVI string into:
      pvi_party  : 'R', 'D', or 'EVEN'
      pvi_score  : signed int (R+7 → +7, D+5 → -5, EVEN → 0)
      pvi_string : clean normalized string e.g. 'R+7'
    """
    pvi_str = str(pvi_str).strip().upper()

    if pvi_str in ("EVEN", "D+0", "R+0", "0", ""):
        return "EVEN", 0, "EVEN"

    m = re.match(r"([RD])\+?(\d+)", pvi_str)
    if not m:
        return pd.NA, pd.NA, pd.NA

    party = m.group(1)
    score = int(m.group(2))
    signed = score if party == "R" else -score

    return party, signed, f"{party}+{score}"


# %%


In [6]:


def load_cook_year(year: int, filepath: str) -> pd.DataFrame:
    df = pd.read_csv(filepath, encoding="latin1")
    df.columns = [c.strip() for c in df.columns]

    df = df.dropna(how="all").copy()

    dist_col = find_dist_col(df.columns.tolist())
    pvi_col  = find_pvi_col(df.columns.tolist(), year)

    out = df[[dist_col, pvi_col]].copy()
    out.columns = ["dist_code_raw", "pvi_raw"]

    out["dist_code"] = out["dist_code_raw"].apply(standardize_dist_code)

    parsed = out["pvi_raw"].apply(parse_pvi)
    out["pvi_party"]  = parsed.apply(lambda x: x[0])
    out["pvi_score"]  = parsed.apply(lambda x: x[1])
    out["pvi_string"] = parsed.apply(lambda x: x[2])

    out["cook_year"] = year

    out = out.dropna(subset=["dist_code"]).copy()
    out = out.drop(columns=["dist_code_raw", "pvi_raw"])

    return out[["cook_year", "dist_code", "pvi_party", "pvi_score", "pvi_string"]]


# %%
# Load all years
cook_parts = []
for year, fname in COOK_FILES.items():
    fpath = os.path.join(COOK_DIR, fname)
    try:
        df_year = load_cook_year(year, fpath)
        cook_parts.append(df_year)
        print(f"{year}: {len(df_year)} districts loaded")
    except Exception as e:
        print(f"{year}: FAILED — {e}")


cook_panel = pd.concat(cook_parts, ignore_index=True)

# Fast lookup used at the end for PVI merges
cook_lookup = (
    cook_panel[["cook_year", "dist_code", "pvi_score", "pvi_party", "pvi_string"]]
    .drop_duplicates(["cook_year", "dist_code"])
    .copy()
)



1997: 435 districts loaded
1999: 435 districts loaded
2001: 435 districts loaded
2003: 435 districts loaded
2005: 435 districts loaded
2007: 435 districts loaded
2009: 435 districts loaded
2011: 435 districts loaded
2013: 435 districts loaded
2015: 435 districts loaded
2017: 435 districts loaded
2019: 435 districts loaded
2021: 435 districts loaded
2023: 435 districts loaded
2025: 435 districts loaded


In [7]:

# %%
# ============================================================
# CHANGE 1: State abbreviation lookup + Cook year mapping
#
# PANEL_YEAR_TO_COOK_YEAR logic:
#   Each panel decade maps to the Cook year covering the COGs
#   financial data window for that decade, so the political
#   environment lines up with the financial data period.
#     1990 panel → COGs ~1992-1997 → Cook 1997 (earliest available)
#     2000 panel → COGs ~1997-2002 → Cook 2003
#     2010 panel → COGs ~2007-2012 → Cook 2011
#     2020 panel → COGs ~2017-2022 → Cook 2017
#   Pre-1990 years have no Cook data → NaN is correct.
# ============================================================

STATE_NAME_TO_ABBR = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY",
}

PANEL_YEAR_TO_COOK_YEAR = {
    1950: None,
    1960: None,
    1970: None,
    1980: None,
    1990: 1997,
    2000: 2003,
    2010: 2011,
    2020: 2017,
}


In [8]:



# %%
# ============================================================
# CHANGE 2: Updated split_panel_one_period_exact
#
# Only differences from original:
#   - Builds _dist_code on the district GDF from STATENAME+DISTRICT
#   - Passes _dist_code through the overlay alongside dist_id_col
#   - Captures ALL qualifying districts per place (ranked by share)
#   - Pivots them wide: dist1, dist1_share, dist2, dist2_share, ...
#   - All other logic (CRS align, make_valid, state filter, THRESH,
#     n_districts, is_split) is identical to original.
# ============================================================

def split_panel_one_period_exact(
    places_in: gpd.GeoDataFrame,
    dists_in: gpd.GeoDataFrame,
    *,
    year: int,
    THRESH: float = 0.05,
    place_state_col: str = "STATE_NAME",
    dist_state_col: str = "STATENAME",
    place_id_col: str = "GEOID",
    dist_id_col: str = "ID",
    place_name_col: Optional[str] = None,
    max_districts: int = 5,
) -> pd.DataFrame:

    places = places_in.copy()
    dists  = dists_in.copy()

    # CRS alignment
    if places.crs != dists.crs:
        dists = dists.to_crs(places.crs)

    # Make geometries valid
    places["geometry"] = places.geometry.make_valid()
    dists["geometry"]  = dists.geometry.make_valid()

    # Build dist_code from STATENAME + DISTRICT
    dists["_state_abbr"] = dists[dist_state_col].astype(str).str.strip().map(STATE_NAME_TO_ABBR)
    dists["_dist_num"]   = dists["DISTRICT"].astype(float).astype(int)
    dists["_dist_code"]  = (
        dists["_state_abbr"] + "-" +
        dists["_dist_num"].astype(str).str.zfill(2)
    )
    # At-large states: remap -00 to -01
    dists["_dist_code"] = dists["_dist_code"].str.replace(r"-00$", "-01", regex=True)

    # Restrict to overlapping states
    place_states  = set(places[place_state_col].astype(str).str.strip().unique())
    dist_states   = set(dists[dist_state_col].astype(str).str.strip().unique())
    common_states = sorted(place_states & dist_states)

    places = places[places[place_state_col].astype(str).str.strip().isin(common_states)].copy()
    dists  = dists[dists[dist_state_col].astype(str).str.strip().isin(common_states)].copy()

    out_counts    = []
    out_all_dists = []

    for st, places_st in places.groupby(place_state_col, sort=False):
        st_key   = str(st).strip()
        dists_st = dists[dists[dist_state_col].astype(str).str.strip() == st_key]

        if dists_st.empty or places_st.empty:
            continue

        place_area_map = places_st.set_index(place_id_col).geometry.area

        pieces = gpd.overlay(
            places_st[[place_id_col, "geometry"]],
            dists_st[[dist_id_col, "_dist_code", "geometry"]],
            how="intersection"
        )

        if pieces.empty:
            continue

        pieces["int_area"]   = pieces.geometry.area
        pieces["place_area"] = pieces[place_id_col].map(place_area_map)
        pieces["share"]      = pieces["int_area"] / pieces["place_area"]

        # Apply threshold (same as original)
        pieces = pieces[pieces["share"] >= THRESH].copy()

        # --- original: count distinct districts per place ---
        summ = (
            pieces.groupby(place_id_col)[dist_id_col]
            .nunique()
            .reset_index(name="n_districts")
        )
        out_counts.append(summ)

        # --- new: keep all qualifying districts ranked by share ---
        pieces_sorted = (
            pieces[[place_id_col, "_dist_code", "share"]]
            .sort_values([place_id_col, "share"], ascending=[True, False])
            .copy()
        )
        pieces_sorted["dist_rank"] = (
            pieces_sorted.groupby(place_id_col).cumcount() + 1
        )
        pieces_sorted = pieces_sorted[
            pieces_sorted["dist_rank"] <= max_districts
        ].copy()

        out_all_dists.append(pieces_sorted)

    # Combine counts (unchanged from original)
    counts = pd.concat(out_counts, ignore_index=True) if out_counts else pd.DataFrame(
        columns=[place_id_col, "n_districts"]
    )

    # Pivot all districts wide
    if out_all_dists:
        all_dists_long = pd.concat(out_all_dists, ignore_index=True)
        dist_wide = all_dists_long.pivot(
            index=place_id_col,
            columns="dist_rank",
            values=["_dist_code", "share"]
        )
        dist_wide.columns = [
            f"dist{rank}" if val == "_dist_code" else f"dist{rank}_share"
            for val, rank in dist_wide.columns
        ]
        dist_wide = dist_wide.reset_index()
    else:
        dist_wide = pd.DataFrame(columns=[place_id_col])

    # Build panel (same structure as original)
    base_cols = [place_id_col]
    if place_name_col:
        base_cols.append(place_name_col)

    panel = places[base_cols].drop_duplicates(place_id_col).copy()
    panel = panel.merge(counts,    on=place_id_col, how="left")
    panel = panel.merge(dist_wide, on=place_id_col, how="left")

    panel["n_districts"] = panel["n_districts"].fillna(0).astype(int)
    panel["is_split"]    = (panel["n_districts"] > 1).astype(int)
    panel["year"]        = int(year)

    # Ensure all distN columns exist up to max_districts
    for i in range(1, max_districts + 1):
        if f"dist{i}" not in panel.columns:
            panel[f"dist{i}"]       = pd.NA
            panel[f"dist{i}_share"] = pd.NA

    dist_cols = []
    for i in range(1, max_districts + 1):
        dist_cols += [f"dist{i}", f"dist{i}_share"]

    panel = panel[base_cols + ["year", "n_districts", "is_split"] + dist_cols]

    return panel


# %%

districts_by_year = {
    1950: gdf_1950s_Dist,
    1960: gdf_1960s_Dist,
    1970: gdf_1970s_Dist,
    1980: gdf_1980s_Dist,
    1990: gdf_1990s_Dist,
    2000: gdf_2000s_Dist,
    2010: gdf_2010s_Dist,
    2020: gdf_2020s_Dist,
}


# %%

def build_places_panel_long(
    places_gdf: gpd.GeoDataFrame,
    districts_by_year: Dict[int, gpd.GeoDataFrame],
    *,
    state_name: str,
    name_col_in: str,
    county_col_in: Optional[str] = None,
    id_source_col_in: Optional[str] = None,
    THRESH: float = 0.03,
    target_crs: int = 5070,
    place_state_col: str = "STATE_NAME",
    dist_state_col: str = "STATENAME",
    place_id_col: str = "PLACE_ID",
    dist_id_col: str = "ID",
    place_name_col: Optional[str] = "NAME",
) -> pd.DataFrame:

    gdf = places_gdf.copy()

    gdf[place_state_col] = str(state_name).strip()
    gdf["NAME"] = gdf[name_col_in].astype(str).str.strip()

    if county_col_in is not None and county_col_in in gdf.columns:
        gdf["COUNTY"] = gdf[county_col_in].astype(str).str.strip()
    else:
        gdf["COUNTY"] = None

    if id_source_col_in is not None and id_source_col_in in gdf.columns:
        gdf["PLACE_ID"] = gdf[id_source_col_in].astype(str).str.strip()
    else:
        gdf = gdf.reset_index(drop=True)
        gdf["PLACE_ID"] = (gdf.index + 1).astype(str)

    places_base = gdf.to_crs(target_crs)

    long_parts = []
    for year, dists in districts_by_year.items():
        df_year = split_panel_one_period_exact(
            places_base,
            dists,
            year=year,
            THRESH=THRESH,
            place_state_col=place_state_col,
            dist_state_col=dist_state_col,
            place_id_col=place_id_col,
            dist_id_col=dist_id_col,
            place_name_col=place_name_col,
        )
        long_parts.append(df_year)

    out = pd.concat(long_parts, ignore_index=True)

    out = out.merge(
        places_base[["PLACE_ID", "COUNTY"]].drop_duplicates("PLACE_ID"),
        on="PLACE_ID",
        how="left"
    )

    return out


In [9]:

# %%
# Path to 2016 precinct-level presidential results
path_2016_election = "/Users/mayarabin/Desktop/Thesis Downloads V2/Election Data/dataverse_files/2016-precinct-president.csv"

election_2016 = pd.read_csv(
    path_2016_election,
    encoding="latin1",
    low_memory=False
)

print(election_2016.shape)
election_2016.head(50000)



(1989234, 37)


,year,stage,special,state,state_postal,state_fips,state_icpsr,county_name,county_fips,county_ansi,county_lat,county_long,jurisdiction,precinct,candidate,candidate_normalized,office,district,writein,party,mode,votes,candidate_opensecrets,candidate_wikidata,candidate_party,candidate_last,candidate_first,candidate_middle,candidate_full,candidate_suffix,candidate_nickname,candidate_fec,candidate_fec_name,candidate_google,candidate_govtrack,candidate_icpsr,candidate_maplight
0,2016,gen,False,Alabama,AL,1,41,Autauga County,1001.0,161526.0,32.532237,-86.64644,Autauga,10 JONES COMMUNITY CTR,Hillary Clinton,clinton,US President,statewide,False,democratic,election day,135,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P00003392,"CLINTON, HILLARY RODHAM / TIMOTHY MICHAEL KAINE",NaN,NaN,NaN,NaN
1,2016,gen,False,Alabama,AL,1,41,Autauga County,1001.0,161526.0,32.532237,-86.64644,Autauga,10 JONES COMMUNITY CTR,Gary Johnson,johnson,US President,statewide,False,independent,election day,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P60012234,"JOHNSON, JOHN FITZGERALD MR.",NaN,NaN,NaN,NaN
2,2016,gen,False,Alabama,AL,1,41,Autauga County,1001.0,161526.0,32.532237,-86.64644,Autauga,10 JONES COMMUNITY CTR,Jill Stein,stein,US President,statewide,False,independent,election day,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P20003984,"STEIN, JILL",NaN,NaN,NaN,NaN
3,2016,gen,False,Alabama,AL,1,41,Autauga County,1001.0,161526.0,32.532237,-86.64644,Autauga,10 JONES COMMUNITY CTR,Donald Trump,trump,US President,statewide,False,republican,election day,218,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P80001571,"TRUMP, DONALD J. / MICHAEL R. PENCE",NaN,NaN,NaN,NaN
4,2016,gen,False,Alabama,AL,1,41,Autauga County,1001.0,161526.0,32.532237,-86.64644,Autauga,10 JONES COMMUNITY CTR,[Write-in],in,US President,statewide,True,NaN,election day,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,2016,gen,False,Arkansas,AR,5,42,Cleburne County,5023.0,66983.0,35.566315,-92.05995,Cleburne County,PINEY-027,Jim Hedges,hedges,US President,statewide,False,independent,Absentee,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49996,2016,gen,False,Arkansas,AR,5,42,Cleburne County,5023.0,66983.0,35.566315,-92.05995,Cleburne County,PINEY-027,Jim Hedges,hedges,US President,statewide,False,independent,Early Vote,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49997,2016,gen,False,Arkansas,AR,5,42,Cleburne County,5023.0,66983.0,35.566315,-92.05995,Cleburne County,PINEY-027,Jim Hedges,hedges,US President,statewide,False,independent,Provisional,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49998,2016,gen,False,Arkansas,AR,5,42,Cleburne County,5023.0,66983.0,35.566315,-92.05995,Cleburne County,PINEY-027,Lynn Kahn,kahn,US President,statewide,False,independent,Election Day,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P60005808,"KAHN, LYNN SANDRA",NaN,NaN,NaN,NaN


In [10]:
# Massachusetts

ma_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Mass/townssurvey_shp/TOWNSSURVEY_POLYM.shp"
ma_gdf = gpd.read_file(ma_path)
ma_gdf.head()

ma_cousub = gpd.read_file(
    "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Mass/tl_2025_25_cousub/tl_2025_25_cousub.shp"
)
ma_cousub = ma_cousub.to_crs(ma_gdf.crs)

ma_gdf = ma_gdf.copy()
ma_cousub = ma_cousub.copy()

ma_gdf["geometry"] = ma_gdf.geometry.make_valid()
ma_cousub["geometry"] = ma_cousub.geometry.make_valid()

ov = gpd.overlay(
    ma_gdf[["TOWN_ID", "TOWN", "geometry"]],
    ma_cousub[["GEOID", "NAME", "geometry"]],
    how="intersection"
)
ov["ov_area"] = ov.geometry.area

best = (
    ov.sort_values("ov_area", ascending=False)
      .groupby("TOWN_ID", as_index=False)
      .first()[["TOWN_ID", "GEOID", "NAME", "ov_area"]]
)

ma_gdf_with_geoid = ma_gdf.merge(best[["TOWN_ID", "GEOID"]], on="TOWN_ID", how="left")

print("Missing GEOIDs:", ma_gdf_with_geoid["GEOID"].isna().sum())
print("Rows:", len(ma_gdf_with_geoid), "Unique GEOIDs:", ma_gdf_with_geoid["GEOID"].nunique())

ma_panel_long = build_places_panel_long(
    ma_gdf_with_geoid,
    districts_by_year,
    state_name="Massachusetts",
    name_col_in="TOWN",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(ma_panel_long)
n_split_rows = (ma_panel_long["is_split"] == 1).sum()
n_unified_rows = (ma_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
ma_panel_long.head()


Missing GEOIDs: 0
Rows: 351 Unique GEOIDs: 351
Total rows: 2808 | Split rows: 44 | Unified rows: 2764


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,2501370045,TOLLAND,1950,1,0,MA-01,0.999439,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,2502111315,CANTON,1950,1,0,MA-13,0.999757,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,2500139100,MASHPEE,1950,1,0,MA-09,0.998193,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,2500937560,LYNNFIELD,1950,1,0,MA-07,0.997959,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,2500103690,BARNSTABLE,1950,1,0,MA-09,0.993393,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [11]:
# Vermont

vt_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Vermont/FS_VCGI_OPENDATA_Boundary_BNDHASH_poly_towns_SP_v1_-7648103760469014069/FS_VCGI_OPENDATA_Boundary_BNDHASH_poly_towns_SP_v1.shp"
vt_gdf = gpd.read_file(vt_path)

print(vt_gdf.crs)
print(vt_gdf.columns)
vt_gdf.head()

vt_panel_long = build_places_panel_long(
    vt_gdf,
    districts_by_year,
    state_name="Vermont",
    name_col_in="TOWNNAMEMC",
    county_col_in=None,
    id_source_col_in="TOWNGEOID",
    THRESH=0.03
)

n_rows = len(vt_panel_long)
n_split_rows = (vt_panel_long["is_split"] == 1).sum()
n_unified_rows = (vt_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
vt_panel_long.head()


PROJCS["NAD83 / Vermont",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",42.5],PARAMETER["central_meridian",-72.5],PARAMETER["scale_factor",0.999964285714286],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Index(['FIPS6', 'TOWNNAME', 'TOWNNAMEMC', 'CNTY', 'TOWNGEOID', 'geometry'], dtype='object')
Total rows: 2048 | Split rows: 0 | Unified rows: 2048


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,5000911800,Canaan,1950,1,0,VT-01,0.99678,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,5001127100,Franklin,1950,1,0,VT-01,0.998669,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,5001105425,Berkshire,1950,1,0,VT-01,0.999732,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,5001133025,Highgate,1950,1,0,VT-01,0.99754,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,5001159125,Richford,1950,1,0,VT-01,0.999641,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [12]:
# New Hampshire

nh_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/New Hampshire/City_Town/City_Town.shp"
nh_gdf = gpd.read_file(nh_path)

nh_cousub = gpd.read_file(
    "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/New Hampshire/tl_2025_33_cousub/tl_2025_33_cousub.shp"
)
nh_cousub = nh_cousub.to_crs(nh_gdf.crs)

nh_gdf = nh_gdf.copy()
nh_cousub = nh_cousub.copy()

nh_gdf["geometry"] = nh_gdf.geometry.make_valid()
nh_cousub["geometry"] = nh_cousub.geometry.make_valid()

ov = gpd.overlay(
    nh_gdf[["fips", "name", "geometry"]],
    nh_cousub[["GEOID", "NAME", "geometry"]],
    how="intersection"
)
ov["ov_area"] = ov.geometry.area

best = (
    ov.sort_values("ov_area", ascending=False)
      .groupby("fips", as_index=False)
      .first()[["fips", "GEOID", "name", "ov_area"]]
)

nh_gdf_with_geoid = nh_gdf.merge(best[["fips", "GEOID"]], on="fips", how="left")

print("Missing GEOIDs:", nh_gdf_with_geoid["GEOID"].isna().sum())
print("Rows:", len(nh_gdf_with_geoid), "Unique GEOIDs:", nh_gdf_with_geoid["GEOID"].nunique())

nh_panel_long = build_places_panel_long(
    nh_gdf_with_geoid,
    districts_by_year,
    state_name="New Hampshire",
    name_col_in="name",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(nh_panel_long)
n_split_rows = (nh_panel_long["is_split"] == 1).sum()
n_unified_rows = (nh_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
nh_panel_long.head(50)



Missing GEOIDs: 0
Rows: 259 Unique GEOIDs: 259
Total rows: 2072 | Split rows: 5 | Unified rows: 2067


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,3300761780,Pittsburg,1950,1,0,NH-02,0.99866,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,3300713220,Clarksville,1950,1,0,NH-02,0.999471,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,3300702420,Atkinson & Gilmanton,1950,1,0,NH-02,0.999802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,3300773380,Stewartstown,1950,1,0,NH-02,0.997731,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,3300768500,Second College,1950,1,0,NH-02,0.999509,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
5,3300718420,Dixville,1950,1,0,NH-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
6,3300718340,Dixs Grant,1950,1,0,NH-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
7,3300713780,Colebrook,1950,1,0,NH-02,0.998662,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
8,3300713940,Columbia,1950,1,0,NH-02,0.998551,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
9,3300780740,Wentworths Location,1950,1,0,NH-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [13]:
# Connecticut

ct_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Connecticut/CT_DCF_Town_Boundaries_-7584911209120945448/CT_DCF_Town_Boundaries.shp"
ct_gdf = gpd.read_file(ct_path)

print(ct_gdf.crs)
print(ct_gdf.columns)
ct_gdf.head()

ct_gdf["GEOID"] = (
    ct_gdf["key_"]
      .astype("Int64")
      .astype(str)
      .str.zfill(10)
)
ct_gdf[["key_", "GEOID"]].head()

ct_panel_long = build_places_panel_long(
    ct_gdf,
    districts_by_year,
    state_name="Connecticut",
    name_col_in="town",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(ct_panel_long)
n_split_rows = (ct_panel_long["is_split"] == 1).sum()
n_unified_rows = (ct_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
ct_panel_long.head()


EPSG:2234
Index(['name', 'mcd', 'key_', 'town', 'dcf_office', 'dcf_region', 'Counties',
       'County_Equ', 'geometry'],
      dtype='object')
Total rows: 1352 | Split rows: 197 | Unified rows: 1155


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,0900104720,Bethel,1950,2,1,CT-01,1.0,CT-04,1.0,NaN,NaN,NaN,NaN,NaN,NaN,None
1,0900108070,Bridgeport,1950,2,1,CT-01,0.977821,CT-04,0.977821,NaN,NaN,NaN,NaN,NaN,NaN,None
2,0900177200,Trumbull,1950,2,1,CT-01,1.0,CT-04,1.0,NaN,NaN,NaN,NaN,NaN,NaN,None
3,0900360120,Plainville,1950,2,1,CT-01,1.0,CT-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,None
4,0900108980,Brookfield,1950,2,1,CT-01,1.0,CT-04,0.99699,NaN,NaN,NaN,NaN,NaN,NaN,None


In [14]:
# Rhode Island

ri_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Rhode Island/BND_Municipalities_1997_spf_-3856937655836009532/BND_Municipalities_1997_spf.shp"
ri_gdf = gpd.read_file(ri_path)

ri_gdf_diss = ri_gdf.dissolve(by="NAME", as_index=False)

ri_cousub = gpd.read_file(
    "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Rhode Island/tl_2025_44_cousub/tl_2025_44_cousub.shp"
)
ri_cousub = ri_cousub.to_crs(ri_gdf_diss.crs)

ri_gdf_diss = ri_gdf_diss.copy()
ri_cousub = ri_cousub.copy()

ri_gdf_diss["geometry"] = ri_gdf_diss.geometry.make_valid()
ri_cousub["geometry"] = ri_cousub.geometry.make_valid()

ri_gdf_diss = ri_gdf_diss.copy()
ri_gdf_diss["PLACE_ID"] = np.arange(len(ri_gdf_diss))

ov = gpd.overlay(
    ri_gdf_diss[["PLACE_ID", "NAME", "geometry"]],
    ri_cousub[["GEOID", "NAME", "geometry"]],
    how="intersection"
)
ov["ov_area"] = ov.geometry.area

best = (
    ov.sort_values("ov_area", ascending=False)
      .groupby("PLACE_ID", as_index=False)
      .first()[["PLACE_ID", "GEOID", "ov_area"]]
)

ri_gdf_with_geoid = ri_gdf_diss.merge(best[["PLACE_ID", "GEOID"]], on="PLACE_ID", how="left")

ri_panel_long = build_places_panel_long(
    ri_gdf_with_geoid,
    districts_by_year,
    state_name="Rhode Island",
    name_col_in="NAME",
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(ri_panel_long)
n_split_rows = (ri_panel_long["is_split"] == 1).sum()
n_unified_rows = (ri_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
ri_panel_long.head()


Total rows: 312 | Split rows: 8 | Unified rows: 304


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,4400105140,BARRINGTON,1950,1,0,RI-01,0.983797,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,4400109280,BRISTOL,1950,1,0,RI-01,0.992245,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,4400711800,BURRILLVILLE,1950,1,0,RI-02,0.99961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,4400714140,CENTRAL FALLS,1950,1,0,RI-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,4400914500,CHARLESTOWN,1950,1,0,RI-02,0.999982,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [15]:
# Maine

me_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Maine/Maine_Town_and_Townships_Boundary_Polygons_-5924039825948470999/Maine_Town_and_Townships_Boundary_Polygons_Feature.shp"
me_gdf = gpd.read_file(me_path)

me_gdf_diss = me_gdf.dissolve(by="TOWN", as_index=False)

me_cousub = gpd.read_file(
    "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Maine/tl_2025_23_cousub/tl_2025_23_cousub.shp"
)
me_cousub = me_cousub.to_crs(me_gdf_diss.crs)

me_gdf_diss = me_gdf_diss.copy()
me_cousub = me_cousub.copy()

me_gdf_diss["geometry"] = me_gdf_diss.geometry.make_valid()
me_cousub["geometry"] = me_cousub.geometry.make_valid()

me_gdf_diss = me_gdf_diss.copy()
me_gdf_diss["PLACE_ID"] = np.arange(len(me_gdf_diss))

ov = gpd.overlay(
    me_gdf_diss[["PLACE_ID", "TOWN", "geometry"]],
    me_cousub[["GEOID", "NAME", "geometry"]],
    how="intersection"
)
ov["ov_area"] = ov.geometry.area

best = (
    ov.sort_values("ov_area", ascending=False)
      .groupby("PLACE_ID", as_index=False)
      .first()[["PLACE_ID", "GEOID", "TOWN", "ov_area"]]
)

me_gdf_with_geoid = me_gdf_diss.merge(best[["PLACE_ID", "GEOID"]], on="PLACE_ID", how="left")

assert me_gdf_with_geoid["GEOID"].isna().sum() == 0

geoid_to_name = me_cousub[["GEOID", "NAME"]].drop_duplicates()

me_one_per_geoid = (
    me_gdf_with_geoid
      .dissolve(by="GEOID", as_index=False)
      .merge(geoid_to_name, on="GEOID", how="left")
)

print("Unique GEOID now?", me_one_per_geoid["GEOID"].is_unique)
print("Rows:", len(me_one_per_geoid))

me_panel_long = build_places_panel_long(
    me_one_per_geoid,
    districts_by_year,
    state_name="Maine",
    name_col_in="NAME",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(me_panel_long)
n_split_rows = (me_panel_long["is_split"] == 1).sum()
n_unified_rows = (me_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
me_panel_long.head(50)


Unique GEOID now? True
Rows: 527
Total rows: 4216 | Split rows: 0 | Unified rows: 4216


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,2300102060,Auburn,1950,1,0,ME-02,0.999958,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,2300119105,Durham,1950,1,0,ME-02,0.997735,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,2300129255,Greene,1950,1,0,ME-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,2300138565,Leeds,1950,1,0,ME-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,2300138740,Lewiston,1950,1,0,ME-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
5,2300140035,Lisbon,1950,1,0,ME-02,0.998187,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
6,2300140665,Livermore,1950,1,0,ME-02,0.999739,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
7,2300140770,Livermore Falls,1950,1,0,ME-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
8,2300144585,Mechanic Falls,1950,1,0,ME-02,0.999386,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
9,2300146160,Minot,1950,1,0,ME-02,0.999331,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [16]:
# New Jersey

nj_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/New Jersey/NJ_Municipalities_3857_-4683990186281872151/NJ_Municipalities_3857.shp"
nj_gdf = gpd.read_file(nj_path).copy()

nj_place = gpd.read_file(
    "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/New Jersey/tl_2025_34_cousub/tl_2025_34_cousub.shp"
).copy()

ea = "EPSG:5070"
nj_gdf   = nj_gdf.to_crs(ea)
nj_place = nj_place.to_crs(ea)

nj_gdf["geometry"]   = nj_gdf.geometry.make_valid()
nj_place["geometry"] = nj_place.geometry.make_valid()

nj_gdf["PLACE_ID"] = np.arange(len(nj_gdf))

ov = gpd.overlay(
    nj_gdf[["PLACE_ID", "GNIS", "GNIS_NAME", "NAME", "geometry"]],
    nj_place[["GEOID", "NAME", "geometry"]],
    how="intersection"
)
ov["ov_area"] = ov.geometry.area

best = (
    ov.sort_values("ov_area", ascending=False)
      .groupby("PLACE_ID", as_index=False)
      .first()[["PLACE_ID", "GEOID", "ov_area"]]
)

nj_gdf_with_geoid = nj_gdf.merge(best[["PLACE_ID", "GEOID"]], on="PLACE_ID", how="left")

print("Missing GEOIDs:", nj_gdf_with_geoid["GEOID"].isna().sum())
print("Rows:", len(nj_gdf_with_geoid), "Unique GEOIDs:", nj_gdf_with_geoid["GEOID"].nunique())
print("Duplicate GEOID rows:", nj_gdf_with_geoid["GEOID"].duplicated(keep=False).sum())

missing = nj_gdf_with_geoid["GEOID"].isna()
if missing.any():
    print(nj_gdf_with_geoid.loc[missing, ["NAME", "GNIS_NAME", "GNIS", "MUN_TYPE"]].head(30))

nj_panel_long = build_places_panel_long(
    nj_gdf_with_geoid,
    districts_by_year,
    state_name="New Jersey",
    name_col_in="NAME",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(nj_panel_long)
n_split_rows = (nj_panel_long["is_split"] == 1).sum()
n_unified_rows = (nj_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
nj_panel_long.head()



Missing GEOIDs: 0
Rows: 564 Unique GEOIDs: 564
Duplicate GEOID rows: 0
Total rows: 4512 | Split rows: 139 | Unified rows: 4373


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,3400910330,Cape May Point Borough,1950,1,0,NJ-02,0.983938,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,3400978530,West Cape May Borough,1950,1,0,NJ-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,3400910270,Cape May,1950,1,0,NJ-02,0.854051,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,3400981200,Wildwood Crest Borough,1950,1,0,NJ-02,0.789471,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,3400980210,West Wildwood Borough,1950,1,0,NJ-02,0.943845,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [17]:
# Pennsylvania

pa_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Pennsylvania/municipalboundaries/municipalboundaries.shp"
pa_gdf = gpd.read_file(pa_path).copy()

pa_gdf.head(30)

pa_gdf["GEOID"] = (
    "42" +
    pa_gdf["FIPS_COUNT"].astype(str).str.zfill(3) +
    pa_gdf["FIPS_MUN_C"].astype(str).str.zfill(5)
)

pa_gdf_diss = pa_gdf.dissolve(by="GEOID", as_index=False)

pa_panel_long = build_places_panel_long(
    pa_gdf_diss,
    districts_by_year,
    state_name="Pennsylvania",
    name_col_in="MUNICIPA_1",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(pa_panel_long)
n_split_rows = (pa_panel_long["is_split"] == 1).sum()
n_unified_rows = (pa_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
pa_panel_long.head()



Total rows: 20552 | Split rows: 402 | Unified rows: 20150


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,4200100116,ABBOTTSTOWN,1950,1,0,PA-19,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,4200102928,ARENDTSVILLE,1950,1,0,PA-19,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,4200105536,BENDERSVILLE,1950,1,0,PA-19,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,4200105880,BERWICK,1950,1,0,PA-19,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,4200106296,BIGLERVILLE,1950,1,0,PA-19,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [18]:
# Delaware

de_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Delaware/Delaware_Hundreds/Delaware_Hundreds.shp"
de_gdf = gpd.read_file(de_path).copy()

de_cousub = gpd.read_file(
    "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Delaware/tl_2025_10_cousub/tl_2025_10_cousub.shp"
).copy()

ea = "EPSG:5070"
de_gdf = de_gdf.to_crs(ea)
de_cousub = de_cousub.to_crs(ea)

de_gdf["geometry"] = de_gdf.geometry.make_valid()
de_cousub["geometry"] = de_cousub.geometry.make_valid()

de_gdf = de_gdf.reset_index(drop=True)
de_gdf["FID"] = np.arange(len(de_gdf))

ov = gpd.overlay(
    de_gdf[["FID", "HUNDRED", "geometry"]],
    de_cousub[["GEOID", "NAME", "geometry"]],
    how="intersection"
)
ov["ov_area"] = ov.geometry.area

best = (
    ov.sort_values("ov_area", ascending=False)
      .groupby("FID", as_index=False)
      .first()[["FID", "GEOID", "ov_area"]]
)

de_gdf_with_geoid = de_gdf.merge(best[["FID", "GEOID"]], on="FID", how="left")

print("Missing GEOIDs:", de_gdf_with_geoid["GEOID"].isna().sum())
print("Rows:", len(de_gdf_with_geoid), "Unique GEOIDs:", de_gdf_with_geoid["GEOID"].nunique())
print("Duplicate GEOID rows:", de_gdf_with_geoid["GEOID"].duplicated(keep=False).sum())

de_one_per_geoid = de_gdf_with_geoid.dissolve(by="GEOID", as_index=False)

de_panel_long = build_places_panel_long(
    de_one_per_geoid,
    districts_by_year,
    state_name="Delaware",
    name_col_in="HUNDRED",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(de_panel_long)
n_split_rows = (de_panel_long["is_split"] == 1).sum()
n_unified_rows = (de_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
de_panel_long.head()


Missing GEOIDs: 0
Rows: 35 Unique GEOIDs: 24
Duplicate GEOID rows: 19
Total rows: 192 | Split rows: 0 | Unified rows: 192


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,1000190444,North Murderkill,1950,1,0,DE-01,0.999967,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,1000190740,Little Creek,1950,1,0,DE-01,0.999541,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,1000190888,South Murderkill,1950,1,0,DE-01,0.999804,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,1000191332,Mispillion,1950,1,0,DE-01,0.999648,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,1000191480,Kenton,1950,1,0,DE-01,0.99997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [19]:
# Maryland

md_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Maryland/Municipal_Boundaries_-_Detailed/Municipal_Boundaries_-_Detailed.shp"
md_gdf = gpd.read_file(md_path)
md_gdf.head()

md_cousub = gpd.read_file(
    "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Maryland/tl_2025_24_cousub/tl_2025_24_cousub.shp"
)
md_cousub = md_cousub.to_crs(md_gdf.crs)

md_gdf = md_gdf.copy()
md_cousub = md_cousub.copy()

md_gdf["geometry"] = md_gdf.geometry.make_valid()
md_cousub["geometry"] = md_cousub.geometry.make_valid()

ov = gpd.overlay(
    md_gdf[["OBJECTID", "MUN_NAME", "geometry"]],
    md_cousub[["GEOID", "NAME", "geometry"]],
    how="intersection"
)
ov["ov_area"] = ov.geometry.area

best = (
    ov.sort_values("ov_area", ascending=False)
      .groupby("OBJECTID", as_index=False)
      .first()[["OBJECTID", "GEOID", "MUN_NAME", "ov_area"]]
)

md_gdf_with_geoid = md_gdf.merge(best[["OBJECTID", "GEOID"]], on="OBJECTID", how="left")

print("Missing GEOIDs:", md_gdf_with_geoid["GEOID"].isna().sum())
print("Rows:", len(md_gdf_with_geoid), "Unique GEOIDs:", md_gdf_with_geoid["GEOID"].nunique())

md_one_per_geoid = md_gdf_with_geoid.dissolve(by="GEOID", as_index=False)

md_panel_long = build_places_panel_long(
    md_one_per_geoid,
    districts_by_year,
    state_name="Maryland",
    name_col_in="MUN_NAME",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(md_panel_long)
n_split_rows = (md_panel_long["is_split"] == 1).sum()
n_unified_rows = (md_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
md_panel_long.head(50)

Missing GEOIDs: 0
Rows: 2677 Unique GEOIDs: 150
Total rows: 1200 | Split rows: 185 | Unified rows: 1015


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,2400190281,CUMBERLAND,1950,1,0,MD-06,0.999982,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,2400190372,CUMBERLAND,1950,1,0,MD-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,2400190463,CUMBERLAND,1950,1,0,MD-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,2400190647,LUKE,1950,1,0,MD-06,0.964943,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,2400190740,BARTON,1950,1,0,MD-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
5,2400190832,LONACONING,1950,1,0,MD-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
6,2400190926,FROSTBURG,1950,1,0,MD-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
7,2400191012,FROSTBURG,1950,1,0,MD-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
8,2400191380,CUMBERLAND,1950,1,0,MD-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
9,2400191595,MIDLAND,1950,1,0,MD-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [20]:
# West Virginia

wv_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/West Virginia/Incorporated_Places_Census_2021_utm83/Incorporated_Places_Census_2021_utm83.shp"
wv_gdf = gpd.read_file(wv_path)

print(wv_gdf.crs)
print(wv_gdf.columns)
wv_gdf.head()

wv_panel_long = build_places_panel_long(
    wv_gdf,
    districts_by_year,
    state_name="West Virginia",
    name_col_in="NAMELSAD",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(wv_panel_long)
n_split_rows = (wv_panel_long["is_split"] == 1).sum()
n_unified_rows = (wv_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
wv_panel_long.head()



EPSG:26917
Index(['PLACENS', 'GEOID', 'NAMELSAD', 'CLASSFP', 'FUNCSTAT', 'ALAND',
       'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')
Total rows: 1856 | Split rows: 13 | Unified rows: 1843


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,5400364,Addison (Webster Springs) town,1950,1,0,WV-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,5400748,Albright town,1950,1,0,WV-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,5400772,Alderson town,1950,1,0,WV-05,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,5401780,Anawalt town,1950,1,0,WV-05,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,5401900,Anmoore town,1950,1,0,WV-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [21]:
# North Carolina

nc_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/North Carolina/tl_2025_37_place/tl_2025_37_place.shp"
nc_gdf = gpd.read_file(nc_path)

print(nc_gdf.crs)
print(nc_gdf.columns)
nc_gdf.head()

nc_panel_long = build_places_panel_long(
    nc_gdf,
    districts_by_year,
    state_name="North Carolina",
    name_col_in="NAME",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(nc_panel_long)
n_split_rows = (nc_panel_long["is_split"] == 1).sum()
n_unified_rows = (nc_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
nc_panel_long.head()



EPSG:4269
Index(['STATEFP', 'PLACEFP', 'PLACENS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD',
       'LSAD', 'CLASSFP', 'PCICBSA', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER',
       'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_8715/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 164 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_8715/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 147 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_8715/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 117 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_8715/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 99 dropped geometries of 

Total rows: 6208 | Split rows: 365 | Unified rows: 5843


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_8715/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 1481 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,3774020,Wilkesboro,1950,1,0,NC-08,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,3747260,Norman,1950,1,0,NC-08,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,3752660,Pittsboro,1950,1,0,NC-04,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,3761860,Siler City,1950,1,0,NC-04,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,3720580,Elizabeth City,1950,1,0,NC-01,0.998672,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [22]:
# Michigan

mi_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Michigan/Minor_Civil_Division_(Cities_%26_Townships)/Minor_Civil_Division_(Cities_%26_Townships).shp"
mi_gdf = gpd.read_file(mi_path)

print(mi_gdf.crs)
print(mi_gdf.columns)
mi_gdf.head()

mi_gdf["GEOID"] = "26" + mi_gdf["FIPSCode"]

mi_panel_long = build_places_panel_long(
    mi_gdf,
    districts_by_year,
    state_name="Michigan",
    name_col_in="Name",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(mi_panel_long)
n_split_rows = (mi_panel_long["is_split"] == 1).sum()
n_unified_rows = (mi_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
mi_panel_long.head()



EPSG:3078
Index(['OBJECTID', 'FIPSCode', 'Name', 'MapLayout', 'FIPSNum', 'Label', 'Type',
       'Peninsula', 'MGFVersion', 'Shape__Are', 'Shape__Len', 'GlobalID',
       'geometry'],
      dtype='object')
Total rows: 12168 | Split rows: 1643 | Unified rows: 10525


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,2687840,Windsor,1950,2,1,MI-01,1.0,MI-03,0.999649,NaN,NaN,NaN,NaN,NaN,NaN,None
1,2670540,Saginaw,1950,2,1,MI-01,1.0,MI-08,1.0,NaN,NaN,NaN,NaN,NaN,NaN,None
2,2674460,Solon,1950,2,1,MI-01,1.0,MI-05,1.0,NaN,NaN,NaN,NaN,NaN,NaN,None
3,2660520,Oliver,1950,2,1,MI-01,1.0,MI-07,1.0,NaN,NaN,NaN,NaN,NaN,NaN,None
4,2635200,Greenwood,1950,2,1,MI-01,1.0,MI-10,0.999672,NaN,NaN,NaN,NaN,NaN,NaN,None


In [23]:
# Wisconsin

wi_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Wisconsin/WI_Cities%2C_Towns%2C_Villages_(July_2025) 2/WI_Cities%2C_Towns%2C_Villages_(July_2025).shp"
wi_gdf = gpd.read_file(wi_path)

print(wi_gdf.crs)
print(wi_gdf.columns)
wi_gdf.head(50)

wi_panel_long = build_places_panel_long(
    wi_gdf,
    districts_by_year,
    state_name="Wisconsin",
    name_col_in="MCD_NAME",
    county_col_in=None,
    id_source_col_in="MCD_FIPS",
    THRESH=0.03
)

n_rows = len(wi_panel_long)
n_split_rows = (wi_panel_long["is_split"] == 1).sum()
n_unified_rows = (wi_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
wi_panel_long.head(50)

EPSG:3070
Index(['OBJECTID', 'MCD_FIPS', 'CNTY_FIPS', 'CNTY_NAME', 'COUSUBFP',
       'MCD_NAME', 'CTV', 'CONTACT', 'DATE_SUB', 'COUNTY', 'PERIOD', 'YEAR',
       'GEOID', 'LABEL', 'Shape__Are', 'Shape__Len', 'geometry'],
      dtype='object')
Total rows: 15304 | Split rows: 141 | Unified rows: 15163


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,5500100275,Adams,1950,1,0,WI-07,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,5500100300,Adams,1950,1,0,WI-07,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,5500107300,Big Flats,1950,1,0,WI-07,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,5500116075,Colburn,1950,1,0,WI-07,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,5500119575,Dell Prairie,1950,1,0,WI-07,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
5,5500122000,Easton,1950,1,0,WI-07,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
6,5500127950,Friendship,1950,1,0,WI-07,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
7,5500137625,Jackson,1950,1,0,WI-07,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
8,5500143425,Leola,1950,1,0,WI-07,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
9,5500144250,Lincoln,1950,1,0,WI-07,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [24]:
# Minnesota

mn_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Minnesota/shp_bdry_mn_city_township_unorg/city_township_unorg.shp"
mn_gdf = gpd.read_file(mn_path)

print(mn_gdf.crs)
print(mn_gdf.columns)
mn_gdf.head(50)

mn_cousub = gpd.read_file(
    "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Minnesota/tl_2025_27_cousub/tl_2025_27_cousub.shp"
)
mn_cousub = mn_cousub.to_crs(mn_gdf.crs)

mn_gdf = mn_gdf.copy()
mn_cousub = mn_cousub.copy()

mn_gdf["geometry"] = mn_gdf.geometry.make_valid()
mn_cousub["geometry"] = mn_cousub.geometry.make_valid()

ov = gpd.overlay(
    mn_gdf[["GNIS_FEATU", "FEATURE_NA", "geometry"]],
    mn_cousub[["GEOID", "NAME", "geometry"]],
    how="intersection"
)
ov["ov_area"] = ov.geometry.area

best = (
    ov.sort_values("ov_area", ascending=False)
      .groupby("GNIS_FEATU", as_index=False)
      .first()[["GNIS_FEATU", "GEOID", "FEATURE_NA", "ov_area"]]
)

mn_gdf_with_geoid = mn_gdf.merge(best[["GNIS_FEATU", "GEOID"]], on="GNIS_FEATU", how="left")

print("Missing GEOIDs:", mn_gdf_with_geoid["GEOID"].isna().sum())
print("Rows:", len(mn_gdf_with_geoid), "Unique GEOIDs:", mn_gdf_with_geoid["GEOID"].nunique())

mn_one_per_geoid = mn_gdf_with_geoid.dissolve(by="GEOID", as_index=False)

mn_panel_long = build_places_panel_long(
    mn_one_per_geoid,
    districts_by_year,
    state_name="Minnesota",
    name_col_in="FEATURE_NA",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(mn_panel_long)
n_split_rows = (mn_panel_long["is_split"] == 1).sum()
n_unified_rows = (mn_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
mn_panel_long.head(150)


EPSG:26915
Index(['GNIS_FEATU', 'FEATURE_NA', 'CTU_CLASS', 'COUNTY_GNI', 'COUNTY_COD',
       'COUNTY_NAM', 'POPULATION', 'SHAPE_Leng', 'SHAPE_Area', 'geometry'],
      dtype='object')
Missing GEOIDs: 0
Rows: 2744 Unique GEOIDs: 2684
Total rows: 21472 | Split rows: 119 | Unified rows: 21353


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,2700100460,Aitkin,1950,1,0,MN-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,2700100478,Aitkin,1950,1,0,MN-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,2700103358,Ball Bluff,1950,1,0,MN-06,0.999919,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,2700103412,Balsam,1950,1,0,MN-06,0.999851,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,2700104384,Beaver,1950,1,0,MN-06,0.999924,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,2700745088,Nebish,1950,1,0,MN-09,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
146,2700746770,North Beltrami,1950,1,0,MN-09,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
147,2700746906,Northern,1950,1,0,MN-09,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
148,2700748040,O'Brien,1950,1,0,MN-09,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Indiana

in_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Indiana/Administrative_Boundaries_of_Indiana_Current/Administrative_Boundaries_of_Indiana_Current.shp"
in_gdf = gpd.read_file(in_path)

print(in_gdf.crs)
print(in_gdf.columns)
in_gdf.head(50)

in_cousub = gpd.read_file(
    "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Indiana/tl_2025_18_cousub/tl_2025_18_cousub.shp"
)
in_cousub = in_cousub.to_crs(in_gdf.crs)

in_gdf = in_gdf.copy()
in_cousub = in_cousub.copy()

in_gdf["geometry"] = in_gdf.geometry.make_valid()
in_cousub["geometry"] = in_cousub.geometry.make_valid()

ov = gpd.overlay(
    in_gdf[["objectid", "name", "geometry"]],
    in_cousub[["GEOID", "NAME", "geometry"]],
    how="intersection"
)
ov["ov_area"] = ov.geometry.area

best = (
    ov.sort_values("ov_area", ascending=False)
      .groupby("objectid", as_index=False)
      .first()[["objectid", "GEOID", "name", "ov_area"]]
)

in_gdf_with_geoid = in_gdf.merge(best[["objectid", "GEOID"]], on="objectid", how="left")

print("Missing GEOIDs:", in_gdf_with_geoid["GEOID"].isna().sum())
print("Rows:", len(in_gdf_with_geoid), "Unique GEOIDs:", in_gdf_with_geoid["GEOID"].nunique())

in_one_per_geoid = in_gdf_with_geoid.dissolve(by="GEOID", as_index=False)

in_panel_long = build_places_panel_long(
    in_one_per_geoid,
    districts_by_year,
    state_name="Indiana",
    name_col_in="name",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(in_panel_long)
n_split_rows = (in_panel_long["is_split"] == 1).sum()
n_unified_rows = (in_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
in_panel_long.head(150)


EPSG:26916
Index(['objectid', 'name', 'type', 'nguid', 'local_id', 'source_dat',
       'source_d_1', 'source_fea', 'source_ori', 'loaddate', 'county_fip',
       'county_id', 'shape_Leng', 'shape_Area', 'geometry'],
      dtype='object')


In [ ]:
# Kansas

ks_cousub = gpd.read_file(
    "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Kansas/tl_2025_20_cousub/tl_2025_20_cousub.shp"
)

print(ks_cousub.crs)
print(ks_cousub.columns)
ks_cousub.head(50)

ks_panel_long = build_places_panel_long(
    ks_cousub,
    districts_by_year,
    state_name="Kansas",
    name_col_in="NAMELSAD",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(ks_panel_long)
n_split_rows = (ks_panel_long["is_split"] == 1).sum()
n_unified_rows = (ks_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
ks_panel_long.head()


EPSG:4269
Index(['STATEFP', 'COUNTYFP', 'COUSUBFP', 'COUSUBNS', 'GEOID', 'GEOIDFQ',
       'NAME', 'NAMELSAD', 'LSAD', 'CLASSFP', 'MTFCC', 'FUNCSTAT', 'ALAND',
       'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')


/Users/mayarabin/Desktop/Thesis Files/.venv/lib/python3.8/site-packages/geopandas/geodataframe.py:1803: FutureWarning: `unary_union` returned None due to all-None GeoSeries. In future, `unary_union` will return 'GEOMETRYCOLLECTION EMPTY' instead.
  merged_geom = block.unary_union
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 1180 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/Users/mayarabin/Desktop/Thesis Files/.venv/lib/python3.8/site-packages/geopandas/geodataframe.py:1803: FutureWarning: `unary_union` returned None due to all-None GeoSeries. In future, `unary_union` will return 'GEOMETRYCOLLECTION EMPTY' instead.
  merged_geom = block.unary_union
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 913 dropped geometri

Total rows: 12168 | Split rows: 36 | Unified rows: 12132


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 1964 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,2004172100,Union township,1950,1,0,KS-04,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,2004177625,Wheatland township,1950,1,0,KS-04,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,2008911750,Center township,1950,1,0,KS-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,2008921575,Erving township,1950,1,0,KS-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,2001153100,Osage township,1950,1,0,KS-02,0.999446,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# California

ca_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/California/California_Cities_and_Identifiers_Blue_Version_view_7320417020940626409/California_City_Boundaries_and_Identifiers.shp"
ca_gdf = gpd.read_file(ca_path)

print(ca_gdf.crs)
print(ca_gdf.columns)
ca_gdf.head(50)

ca_one_per_geoid = ca_gdf.dissolve(by="CENSUS_GEO", as_index=False)

ca_panel_long = build_places_panel_long(
    ca_one_per_geoid,
    districts_by_year,
    state_name="California",
    name_col_in="CDTFA_CITY",
    county_col_in=None,
    id_source_col_in="CENSUS_GEO",
    THRESH=0.03
)

n_rows = len(ca_panel_long)
n_split_rows = (ca_panel_long["is_split"] == 1).sum()
n_unified_rows = (ca_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
ca_panel_long.head(50)



EPSG:3310
Index(['CDTFA_COPR', 'CDTFA_CITY', 'CDTFA_COUN', 'CENSUS_PLA', 'CENSUS_GEO',
       'CENSUS_P_1', 'GNIS_PLACE', 'GNIS_ID', 'CDT_CITY_A', 'CDT_COUNTY',
       'PRIMARY_DO', 'CENSUS_POP', 'CDT_NAME_S', 'OFFSHORE', 'AREA_SQMI',
       'GlobalID', 'geometry'],
      dtype='object')
Total rows: 3856 | Split rows: 468 | Unified rows: 3388


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,0600296,Adelanto,1950,1,0,CA-27,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,0600394,Agoura Hills,1950,1,0,CA-22,0.999941,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,0600562,Alameda,1950,1,0,CA-08,0.472055,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,0600674,Albany,1950,1,0,CA-07,0.336022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,0600884,Alhambra,1950,1,0,CA-25,0.993123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
5,0600947,Aliso Viejo,1950,1,0,CA-28,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
6,0601444,Alturas,1950,1,0,CA-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
7,0601514,Amador City,1950,1,0,CA-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
8,0601640,American Canyon,1950,1,0,CA-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
9,0602000,Anaheim,1950,1,0,CA-28,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Florida

fl_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Florida/par_citylm_2021/par_citylm_2021.shp"
fl_gdf = gpd.read_file(fl_path)

print(fl_gdf.crs)
print(fl_gdf.columns)
fl_gdf.head(50)

fl_gdf["STPLACEFP"] = "12" + fl_gdf["PLACEFP"]

fl_panel_long = build_places_panel_long(
    fl_gdf,
    districts_by_year,
    state_name="Florida",
    name_col_in="NAME",
    county_col_in=None,
    id_source_col_in="STPLACEFP",
    THRESH=0.03
)

n_rows = len(fl_panel_long)
n_split_rows = (fl_panel_long["is_split"] == 1).sum()
n_unified_rows = (fl_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
fl_panel_long.head(50)


EPSG:3087
Index(['PLACEFP', 'BEBR_ID', 'NAME', 'COUNTY', 'TAX_COUNT', 'TAXAUTHCD',
       'NOTES', 'SRC_NAME', 'SRC_URL', 'SRC_DATE', 'PCT_CHANGE', 'ACRES',
       'DESCRIPT', 'FGDLAQDATE', 'AUTOID', 'SHAPE_AREA', 'SHAPE_LEN',
       'geometry'],
      dtype='object')
Total rows: 3288 | Split rows: 329 | Unified rows: 2959


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,1208600,BRISTOL,1950,1,0,FL-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,1209725,CALLAWAY,1950,1,0,FL-03,0.993204,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,1209900,CAMPBELLTON,1950,1,0,FL-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,1210725,CARRABELLE,1950,1,0,FL-03,0.991193,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,1210975,CARYVILLE,1950,1,0,FL-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
5,1211800,CHATTAHOOCHEE,1950,1,0,FL-03,0.999983,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
6,1216800,DEFUNIAK SPRINGS,1950,1,0,FL-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
7,1219725,EBRO,1950,1,0,FL-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
8,1221250,ESTO,1950,1,0,FL-03,0.998168,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
9,1211975,CHIPLEY,1950,1,0,FL-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Texas

tx_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Texas/TxDOT_City_Boundaries_-7440712964994907480/Cities.shp"
tx_gdf = gpd.read_file(tx_path)

print(tx_gdf.crs)
print(tx_gdf.columns)
print(tx_gdf.shape)
tx_gdf.head(50)

tx_gdf = tx_gdf.drop(index=296)
tx_gdf = tx_gdf.to_crs(5070)
tx_gdf["geometry"] = tx_gdf.geometry.buffer(0)

STATE_NAME = "Texas"

districts_by_year_tx = {}
for yr, g in districts_by_year.items():
    g2 = g[g["STATENAME"].astype(str).str.strip() == STATE_NAME].copy()
    districts_by_year_tx[yr] = g2
    print(yr, len(g2))

districts_by_year_tx = {
    yr: g[g["DISTRICT"] != 0].copy()
    for yr, g in districts_by_year_tx.items()
}

tx_panel_long = build_places_panel_long(
    tx_gdf,
    districts_by_year_tx,
    state_name="Texas",
    name_col_in="CITY_NM",
    county_col_in=None,
    id_source_col_in="CITY_FIPS",
    THRESH=0.05
)

n_rows = len(tx_panel_long)
n_split_rows = (tx_panel_long["is_split"] == 1).sum()
n_unified_rows = (tx_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
tx_panel_long.head(50)


EPSG:3857
Index(['CITY_NM', 'TXDOT_CITY', 'CITY_FIPS', 'CNTY_SEAT_', 'POP1990',
       'POP2000', 'POP2010', 'POP2020', 'POP2022', 'POP_CD', 'MAP_COLOR_',
       'COLOR_CD', 'GID', 'geometry'],
      dtype='object')
(1226, 14)
1950 22
1960 23
1970 24
1980 27
1990 30
2000 32
2010 36
2020 38
Total rows: 9800 | Split rows: 708 | Unified rows: 9092


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,4819435,Dayton Lakes,1950,1,0,TX-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,4811536,Burton,1950,1,0,TX-10,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,4818476,Daisetta,1950,1,0,TX-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,4838872,Kenefick,1950,1,0,TX-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,4863584,Roxton,1950,1,0,TX-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
5,4820020,Deport,1950,1,0,TX-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
6,4842004,Leakey,1950,1,0,TX-21,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
7,4868756,Sonora,1950,1,0,TX-21,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
8,4818224,Cushing,1950,1,0,TX-07,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
9,4849320,Moran,1950,1,0,TX-17,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Oklahoma

ok_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Oklahoma/Municipal_Boundaries_6223949470727062199 (1)/muni_2020.shp"
ok_gdf = gpd.read_file(ok_path)

print(ok_gdf.crs)
print(ok_gdf.columns)
ok_gdf.head(50)

ok_gdf = ok_gdf.copy()
ok_gdf["PLACEFP"] = (ok_gdf["FIPS"].astype(int).astype(str).str.zfill(5))
ok_gdf["STATEFP"] = "40"
ok_gdf["GEOID"] = ok_gdf["STATEFP"] + ok_gdf["PLACEFP"]
ok_gdf = ok_gdf.dissolve(by="GEOID", as_index=False)

ok_panel_long = build_places_panel_long(
    ok_gdf,
    districts_by_year,
    state_name="Oklahoma",
    name_col_in="CITYNAME",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(ok_panel_long)
n_split_rows = (ok_panel_long["is_split"] == 1).sum()
n_unified_rows = (ok_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
ok_panel_long.head(50)


EPSG:3857
Index(['ST_FIPS', 'CO_FIPS', 'COUNTY', 'CITYNAME', 'FIPS', 'AD_VAL_NUM',
       'LAST_UPDAT', 'LAST_VERIF', 'ACRES', 'geometry'],
      dtype='object')
Total rows: 4776 | Split rows: 141 | Unified rows: 4635


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,4000100,Achille,1950,1,0,OK-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,4000200,Ada,1950,1,0,OK-04,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,4000250,Adair,1950,1,0,OK-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,4000450,Addington,1950,1,0,OK-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,4000600,Afton,1950,1,0,OK-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
5,4000700,Agra,1950,1,0,OK-04,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
6,4001050,Albion,1950,1,0,OK-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
7,4001150,Alderson,1950,1,0,OK-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
8,4001250,Alex,1950,1,0,OK-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
9,4001350,Aline,1950,1,0,OK-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Nebraska

ne_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Nebraska/BND_MunicipalBoundaries_OCIO/BND_MunicipalBoundaries_OCIO.shp"
ne_gdf = gpd.read_file(ne_path)

print(ne_gdf.crs)
print(ne_gdf.columns)
ne_gdf.head(50)

ne_panel_long = build_places_panel_long(
    ne_gdf,
    districts_by_year,
    state_name="Nebraska",
    name_col_in="NAMELSAD",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(ne_panel_long)
n_split_rows = (ne_panel_long["is_split"] == 1).sum()
n_unified_rows = (ne_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
ne_panel_long.head(50)


EPSG:3857
Index(['OBJECTID', 'STATEFP', 'PLACEFP', 'PLACENS', 'GEOID', 'NAME',
       'NAMELSAD', 'LSAD', 'CLASSFP', 'PCICBSA', 'PCINECTA', 'MTFCC',
       'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'COUNTYSEAT',
       'Shape_Leng', 'Shape_Area', 'geometry'],
      dtype='object')
Total rows: 4232 | Split rows: 13 | Unified rows: 4219


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,3112105,Danbury village,1950,1,0,NE-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,3103180,Bartley village,1950,1,0,NE-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,3123830,Indianola city,1950,1,0,NE-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,3102620,Atlanta village,1950,1,0,NE-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,3120470,Hadar village,1950,1,0,NE-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
5,3121415,Hastings city,1950,1,0,NE-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
6,3103775,Belden village,1950,1,0,NE-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
7,3109865,Coleridge village,1950,1,0,NE-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
8,3117110,Fordyce village,1950,1,0,NE-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
9,3121275,Hartington city,1950,1,0,NE-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Colorado

co_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Colorado/DOLA_Municipalities/DOLA_Municipalities.shp"
co_gdf = gpd.read_file(co_path)

print(co_gdf.crs)
print(co_gdf.columns)
co_gdf.head(50)

co_gdf["STPLACEFP"] = ("08" + co_gdf["city"].astype(str)).astype(str)
co_gdf = co_gdf.dissolve(by="STPLACEFP", as_index=False)

co_panel_long = build_places_panel_long(
    co_gdf,
    districts_by_year,
    state_name="Colorado",
    name_col_in="cityname",
    county_col_in=None,
    id_source_col_in="STPLACEFP",
    THRESH=0.03
)

n_rows = len(co_panel_long)
n_split_rows = (co_panel_long["is_split"] == 1).sum()
n_unified_rows = (co_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")


EPSG:3857
Index(['gid', 'city', 'cityname', 'rec_num', 'county', 'cl_re_date', 'descr',
       'ord_num', 'type', 'notes', 'OBJECTID', 'geometry'],
      dtype='object')
Total rows: 2184 | Split rows: 86 | Unified rows: 2098


In [ ]:
# Utah

ut_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Utah/UtahMunicipalBoundaries_-4816321826181792416/Municipalities.shp"
ut_gdf = gpd.read_file(ut_path)

print(ut_gdf.crs)
print(ut_gdf.columns)
ut_gdf.head(50)

ut_gdf["STPLACEFP"] = ("49" + ut_gdf["FIPS"].astype(str)).astype(str)
ut_gdf["geometry"] = ut_gdf.geometry.buffer(0)
ut_gdf = ut_gdf.dissolve(by=["STPLACEFP", "NAME"], as_index=False)
ut_gdf = ut_gdf[ut_gdf.geometry.notna()].copy()

ut_panel_long = build_places_panel_long(
    ut_gdf,
    districts_by_year,
    state_name="Utah",
    name_col_in="NAME",
    county_col_in=None,
    id_source_col_in="STPLACEFP",
    THRESH=0.03
)

n_rows = len(ut_panel_long)
n_split_rows = (ut_panel_long["is_split"] == 1).sum()
n_unified_rows = (ut_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
ut_panel_long.head()



EPSG:3857
Index(['COUNTYNBR', 'NAME', 'COUNTYSEAT', 'SHORTDESC', 'UPDATED', 'FIPS',
       'ENTITYNBR', 'SALESTAXID', 'IMSCOLOR', 'MINNAME', 'POPLASTCEN',
       'POPLASTEST', 'UGRCODE', 'GNIS', 'GlobalID', 'geometry'],
      dtype='object')


/Users/mayarabin/Desktop/Thesis Files/.venv/lib/python3.8/site-packages/geopandas/geodataframe.py:1803: FutureWarning: `unary_union` returned None due to all-None GeoSeries. In future, `unary_union` will return 'GEOMETRYCOLLECTION EMPTY' instead.
  merged_geom = block.unary_union


Total rows: 1992 | Split rows: 55 | Unified rows: 1937


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,4900540,Alpine,1950,1,0,UT-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,4900650,Alta,1950,1,0,UT-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,4900760,Altamont,1950,1,0,UT-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,4900870,Alton,1950,1,0,UT-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,4901090,Amalga,1950,1,0,UT-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Arizona

az_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Arizona/Incorporated_Places_Boundaries_8172707889525712720/Incorporated_Places_Boundaries.shp"
az_gdf = gpd.read_file(az_path)

print(az_gdf.crs)
print(az_gdf.columns)

az_gdf["PLACEFP"] = (az_gdf["FIPS"].astype(int).astype(str).str.zfill(7))
az_gdf.head(50)

az_panel_long = build_places_panel_long(
    az_gdf,
    districts_by_year,
    state_name="Arizona",
    name_col_in="Name",
    county_col_in=None,
    id_source_col_in="PLACEFP",
    THRESH=0.03
)

n_rows = len(az_panel_long)
n_split_rows = (az_panel_long["is_split"] == 1).sum()
n_unified_rows = (az_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")



EPSG:4326
Index(['Name', 'Region_Cod', 'FIPS', 'AreaType', 'POP', 'POPDATE', 'PCT_CHG',
       'POP2020', 'POP2021', 'POP2022', 'POP2023', 'POP2024', 'SQ_Miles',
       'SQ_Acres', 'geometry'],
      dtype='object')
Total rows: 728 | Split rows: 87 | Unified rows: 641


In [ ]:
# Nevada

nv_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Nevada/tl_2025_32_place/tl_2025_32_place.shp"
nv_gdf = gpd.read_file(nv_path)

print(nv_gdf.crs)
print(az_gdf.columns)
nv_gdf.head(50)

nv_panel_long = build_places_panel_long(
    nv_gdf,
    districts_by_year,
    state_name="Nevada",
    name_col_in="NAME",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(nv_panel_long)
n_split_rows = (nv_panel_long["is_split"] == 1).sum()
n_unified_rows = (nv_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
nv_panel_long.head()



EPSG:4269
Index(['Name', 'Region_Cod', 'FIPS', 'AreaType', 'POP', 'POPDATE', 'PCT_CHG',
       'POP2020', 'POP2021', 'POP2022', 'POP2023', 'POP2024', 'SQ_Miles',
       'SQ_Acres', 'geometry', 'PLACEFP'],
      dtype='object')


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 32 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(


Total rows: 1104 | Split rows: 34 | Unified rows: 1070


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 193 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,3260600,Reno,1950,1,0,NV-01,0.99999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,3208500,Caliente,1950,1,0,NV-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,3223500,Ely,1950,1,0,NV-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,3284800,Winnemucca,1950,1,0,NV-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,3206500,Boulder City,1950,1,0,NV-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# New Mexico

nm_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/New Mexico/tl_2025_35_place/tl_2025_35_place.shp"
nm_gdf = gpd.read_file(nm_path)

print(nm_gdf.crs)
print(nm_gdf.columns)
nm_gdf.head(50)

nm_panel_long = build_places_panel_long(
    nm_gdf,
    districts_by_year,
    state_name="New Mexico",
    name_col_in="NAME",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(nm_panel_long)
n_split_rows = (nm_panel_long["is_split"] == 1).sum()
n_unified_rows = (nm_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
nm_panel_long.head()



EPSG:4269
Index(['STATEFP', 'PLACEFP', 'PLACENS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD',
       'LSAD', 'CLASSFP', 'PCICBSA', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER',
       'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 6 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 24 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 27 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 160 dropped geometries of dif

Total rows: 4224 | Split rows: 59 | Unified rows: 4165


/Users/mayarabin/Desktop/Thesis Files/.venv/lib/python3.8/site-packages/geopandas/geodataframe.py:1803: FutureWarning: `unary_union` returned None due to all-None GeoSeries. In future, `unary_union` will return 'GEOMETRYCOLLECTION EMPTY' instead.
  merged_geom = block.unary_union
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 136 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,3525450,Eunice,1950,1,0,NM-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,3544490,Lovington,1950,1,0,NM-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,3576620,Tatum,1950,1,0,NM-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,3506480,Belen,1950,1,0,NM-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,3527340,Fort Sumner,1950,1,0,NM-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Oregon

or_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Oregon/City_Limits/City_Limits.shp"
or_gdf = gpd.read_file(or_path)

print(or_gdf.crs)
print(or_gdf.columns)

or_gdf["STPLACEFP"] = ("41" + or_gdf["instCode"].astype(str)).astype(str)
or_gdf.head(50)

or_panel_long = build_places_panel_long(
    or_gdf,
    districts_by_year,
    state_name="Oregon",
    name_col_in="CITYNAME",
    county_col_in=None,
    id_source_col_in="STPLACEFP",
    THRESH=0.03
)

n_rows = len(or_panel_long)
n_split_rows = (or_panel_long["is_split"] == 1).sum()
n_unified_rows = (or_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
or_panel_long.head()


EPSG:3857
Index(['OBJECTID', 'CITYNAME', 'descriptn', 'instCode', 'unitOwner', 'acres',
       'EFFECTV_DT', 'GIS_PRC_DT', 'geometry'],
      dtype='object')
Total rows: 1928 | Split rows: 79 | Unified rows: 1849


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,4100275,Adair Village,1950,1,0,OR-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,4100350,Adams,1950,1,0,OR-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,4100500,Adrian,1950,1,0,OR-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,4101000,Albany,1950,2,1,OR-04,0.777893,OR-01,0.222107,NaN,NaN,NaN,NaN,NaN,NaN,None
4,4102000,Amity,1950,1,0,OR-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Washington

wa_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Washington/WSDOT_-_City_Limits/WSDOT_-_City_Limits.shp"
wa_gdf = gpd.read_file(wa_path)

print(wa_gdf.crs)
print(wa_gdf.columns)
wa_gdf.head(50)

STATE_NAME = "Washington"

districts_by_year_wa = {}
for yr, g in districts_by_year.items():
    g2 = g[g["STATENAME"].astype(str).str.strip() == STATE_NAME].copy()
    districts_by_year_wa[yr] = g2
    print(yr, len(g2))

districts_by_year_wa = {
    yr: g[g["DISTRICT"] != 0].copy()
    for yr, g in districts_by_year_wa.items()
}

wa_panel_long = build_places_panel_long(
    wa_gdf,
    districts_by_year_wa,
    state_name="Washington",
    name_col_in="CityName",
    county_col_in=None,
    id_source_col_in="CityFIPSLo",
    THRESH=0.03
)

n_rows = len(wa_panel_long)
n_split_rows = (wa_panel_long["is_split"] == 1).sum()
n_unified_rows = (wa_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
wa_panel_long.head(50)



EPSG:3857
Index(['OBJECTID', 'CityName', 'CountySeat', 'CityGNISPl', 'LastUpdate',
       'CountyFIPS', 'MajorCity', 'CityFIPSLo', 'OFMCityCod', 'GlobalID',
       'SHAPESTAre', 'SHAPESTLen', 'geometry'],
      dtype='object')
1950 7
1960 7
1970 7
1980 10
1990 10
2000 9
2010 10
2020 10
Total rows: 2248 | Split rows: 104 | Unified rows: 2144


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,5337270,Lake Forest Park,1950,1,0,WA-02,0.99117,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,5333175,Index,1950,1,0,WA-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,5310495,Cashmere,1950,1,0,WA-05,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,5307380,Bothell,1950,1,0,WA-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,5362288,SeaTac,1950,1,0,WA-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
5,5313890,Colton,1950,1,0,WA-04,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
6,5301010,Albion,1950,1,0,WA-04,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
7,5318965,DuPont,1950,1,0,WA-06,0.989682,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
8,5312630,Clarkston,1950,1,0,WA-04,0.99951,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
9,5336780,La Conner,1950,1,0,WA-02,0.938835,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Idaho

id_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Idaho/tl_2025_16_place/tl_2025_16_place.shp"
id_gdf = gpd.read_file(id_path)

print(id_gdf.crs)
print(id_gdf.columns)
id_gdf.head(50)

id_panel_long = build_places_panel_long(
    id_gdf,
    districts_by_year,
    state_name="Idaho",
    name_col_in="NAMELSAD",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(id_panel_long)
n_split_rows = (id_panel_long["is_split"] == 1).sum()
n_unified_rows = (id_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
id_panel_long.head()



EPSG:4269
Index(['STATEFP', 'PLACEFP', 'PLACENS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD',
       'LSAD', 'CLASSFP', 'PCICBSA', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER',
       'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 49 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 49 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 49 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 49 dropped geometries of dif

Total rows: 1888 | Split rows: 20 | Unified rows: 1868


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 49 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,1645910,Leadore city,1950,1,0,ID-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,1652030,Menan city,1950,1,0,ID-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,1655450,Mud Lake city,1950,1,0,ID-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,1667780,Rigby city,1950,1,0,ID-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,1667960,Ririe city,1950,1,0,ID-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Montana

mt_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Montana/MontanaIncorporatedCitiesTowns_shp/MontanaIncorporatedCitiesTowns.shp"
mt_gdf = gpd.read_file(mt_path)

print(mt_gdf.crs)
print(mt_gdf.columns)
mt_gdf.head(50)

mt_gdf["PLACEFP"] = mt_gdf["FIPS_CODE"].astype("Int64").astype(str)

STATE_NAME = "Montana"

districts_by_year_mt = {}
for yr, g in districts_by_year.items():
    g2 = g[g["STATENAME"].astype(str).str.strip() == STATE_NAME].copy()
    districts_by_year_mt[yr] = g2
    print(yr, len(g2))

districts_by_year_mt = {
    yr: g[g["DISTRICT"] != 0].copy()
    for yr, g in districts_by_year_mt.items()
}

mt_panel_long = build_places_panel_long(
    mt_gdf,
    districts_by_year_mt,
    state_name="Montana",
    name_col_in="NAME",
    county_col_in=None,
    id_source_col_in="PLACEFP",
    THRESH=0.03
)

n_rows = len(mt_panel_long)
n_split_rows = (mt_panel_long["is_split"] == 1).sum()
n_unified_rows = (mt_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
mt_panel_long.head()



EPSG:6514
Index(['NAME', 'COUNTY', 'METADATA', 'FIPS_CODE', 'LAST_UPDAT', 'ENTITY_ID',
       'DATASET_ID', 'PKEY', 'CLASS', 'TYPE', 'YEAR', 'Acres', 'SqMiles',
       'BAS_ID', 'ID_UK', 'City_Town', 'DistrictCo', 'COUNTYNAME',
       'Shape_Leng', 'Shape_Area', 'geometry'],
      dtype='object')
1950 3
1960 3
1970 3
1980 3
1990 1
2000 1
2010 1
2020 2
Total rows: 645 | Split rows: 0 | Unified rows: 645


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,3079525,West Yellowstone,1950,1,0,MT-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,3043525,Lima,1950,1,0,MT-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,3009700,Broadus,1950,1,0,MT-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,3016525,Colstrip,1950,1,0,MT-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,3023650,Ekalaka,1950,1,0,MT-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# South Dakota

sd_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/South Dakota/tl_2025_46_place/tl_2025_46_place.shp"
sd_gdf = gpd.read_file(sd_path)

print(sd_gdf.crs)
print(sd_gdf.columns)
sd_gdf.head(50)

STATE_NAME = "South Dakota"

districts_by_year_sd = {}
for yr, g in districts_by_year.items():
    g2 = g[g["STATENAME"].astype(str).str.strip() == STATE_NAME].copy()
    districts_by_year_sd[yr] = g2
    print(yr, len(g2))

districts_by_year_sd = {
    yr: g[g["DISTRICT"] != 0].copy()
    for yr, g in districts_by_year_sd.items()
}

sd_panel_long = build_places_panel_long(
    sd_gdf,
    districts_by_year_sd,
    state_name="South Dakota",
    name_col_in="NAMELSAD",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(sd_panel_long)
n_split_rows = (sd_panel_long["is_split"] == 1).sum()
n_unified_rows = (sd_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
sd_panel_long.head()


EPSG:4269
Index(['STATEFP', 'PLACEFP', 'PLACENS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD',
       'LSAD', 'CLASSFP', 'PCICBSA', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER',
       'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')
1950 3
1960 3
1970 3
1980 1
1990 1
2000 1
2010 1
2020 1
Total rows: 1455 | Split rows: 2 | Unified rows: 1453


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,4616620,Dimock town,1950,1,0,SD-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,4641980,Menno city,1950,1,0,SD-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,4646900,Olivet town,1950,1,0,SD-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,4648460,Parkston city,1950,1,0,SD-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,4608340,Buffalo Gap town,1950,1,0,SD-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# North Dakota

nd_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/North Dakota/NDGISHUB_City_Boundaries_6280708421095695401/Corporate_Boundaries.shp"
nd_gdf = gpd.read_file(nd_path)

print(nd_gdf.crs)
print(nd_gdf.columns)

nd_gdf["STPLACEFP"] = "38" + nd_gdf["CITY_ID"].astype(str).str.zfill(5)
nd_gdf.head(50)

nd_panel_long = build_places_panel_long(
    nd_gdf,
    districts_by_year,
    state_name="North Dakota",
    name_col_in="CITY_NAME",
    county_col_in=None,
    id_source_col_in="STPLACEFP",
    THRESH=0.03
)

n_rows = len(nd_panel_long)
n_split_rows = (nd_panel_long["is_split"] == 1).sum()
n_unified_rows = (nd_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
nd_panel_long.head()



EPSG:3857
Index(['SOURCE_ID', 'CITY_NAME', 'CITY_ID', 'CNTY_SEAT', 'CREATED_US',
       'CREATED_DA', 'LAST_EDITE', 'LAST_EDI_1', 'HPMS_URBAN', 'geometry'],
      dtype='object')
Total rows: 2848 | Split rows: 356 | Unified rows: 2492


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,3816940,Crosby,1950,1,0,ND-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,3863740,Portal,1950,1,0,ND-01,0.996894,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,3801860,Ambrose,1950,1,0,ND-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,3827820,Fortuna,1950,1,0,ND-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,3855620,Neche,1950,1,0,ND-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Wyoming

wy_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Wyoming/tl_2025_56_place/tl_2025_56_place.shp"
wy_gdf = gpd.read_file(wy_path)

print(wy_gdf.crs)
print(wy_gdf.columns)
wy_gdf.head(50)

wy_panel_long = build_places_panel_long(
    wy_gdf,
    districts_by_year,
    state_name="Wyoming",
    name_col_in="NAMELSAD",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(wy_panel_long)
n_split_rows = (wy_panel_long["is_split"] == 1).sum()
n_unified_rows = (wy_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
wy_panel_long.head()


EPSG:4269
Index(['STATEFP', 'PLACEFP', 'PLACENS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD',
       'LSAD', 'CLASSFP', 'PCICBSA', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER',
       'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')
Total rows: 1640 | Split rows: 0 | Unified rows: 1640


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,5642730,Kirby town,1950,1,0,WY-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,5676515,Thermopolis town,1950,1,0,WY-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,5685015,Wright town,1950,1,0,WY-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,5675790,Ten Sleep town,1950,1,0,WY-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,5620690,Dixon town,1950,1,0,WY-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Louisiana

la_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Louisiana/Municipal_Boundaries/Municipal_Boundaries.shp"
la_gdf = gpd.read_file(la_path)

print(la_gdf.crs)
print(la_gdf.columns)
la_gdf.head(50)

la_panel_long = build_places_panel_long(
    la_gdf,
    districts_by_year,
    state_name="Louisiana",
    name_col_in="PlaceNameF",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(la_panel_long)
n_split_rows = (la_panel_long["is_split"] == 1).sum()
n_unified_rows = (la_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
la_panel_long.head()


EPSG:26915
Index(['OBJECTID', 'PlaceFIPS', 'PlaceNameS', 'PlaceNameF', 'PlaceGNISC',
       'PlaceClass', 'PlaceCla_1', 'Functional', 'Function_1', 'GEOID',
       'LegalStati', 'LegalSta_1', 'StatAreaPr', 'StatArea_1', 'CentroidLo',
       'CentroidLa', 'InternalPo', 'Internal_1', 'AreaLandSq', 'AreaWaterS',
       'ParishName', 'ParishNumb', 'ParishFIPS', 'ParishNa_1', 'ParishNu_1',
       'ParishFI_1', 'ParishNa_2', 'ParishNu_2', 'ParishFI_2', 'ParishNa_3',
       'ParishNu_3', 'ParishFI_3', 'ParishNa_4', 'ParishNu_4', 'ParishFI_4',
       'DOTD_Distr', 'DOTD_Dis_1', 'DOTD_Dis_2', 'DOTD_Dis_3', 'AreaAcres',
       'AreaSquare', 'created_us', 'created_da', 'created__1', 'last_edite',
       'last_edi_1', 'last_edi_2', 'service_la', 'service__1', 'GlobalID',
       'Shape_Leng', 'Shape_Area', 'geometry'],
      dtype='object')
Total rows: 2432 | Split rows: 94 | Unified rows: 2338


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,2252075,Morganza village,1950,1,0,LA-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,2208920,Bossier City city,1950,1,0,LA-04,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,2252320,Morse village,1950,1,0,LA-07,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,2243920,Lillie village,1950,1,0,LA-05,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,2212665,Carencro city,1950,1,0,LA-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Arkansas

ar_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Arkansas/MUNICIPAL_BOUNDARY/MUNICIPAL_BOUNDARY.shp"
ar_gdf = gpd.read_file(ar_path)

print(ar_gdf.crs)
print(ar_gdf.columns)

ar_gdf["STPLACEFP"] = "05" + ar_gdf["city_fips"].astype(str).str.zfill(5)

ar_panel_long = build_places_panel_long(
    ar_gdf,
    districts_by_year,
    state_name="Arkansas",
    name_col_in="city_name",
    county_col_in=None,
    id_source_col_in="STPLACEFP",
    THRESH=0.03
)

n_rows = len(ar_panel_long)
n_split_rows = (ar_panel_long["is_split"] == 1).sum()
n_unified_rows = (ar_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
ar_panel_long.head()



EPSG:26915
Index(['objectid', 'city_name', 'city_fips', 'effective_', 'revised_da',
       'revision_t', 'squaremile', 'acres', 'pop2000', 'pop2010', 'pop2020',
       'classifica', 'st_area(sh', 'st_length(', 'geometry'],
      dtype='object')
Total rows: 4008 | Split rows: 33 | Unified rows: 3975


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,0562960,Scranton,1950,1,0,AR-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,0536550,Keo,1950,1,0,AR-06,0.998294,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,0551500,O'Kean,1950,1,0,AR-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,0523800,Fisher,1950,1,0,AR-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,0504840,Bella Vista,1950,1,0,AR-03,0.998589,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Mississippi

ms_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Mississippi/tl_2025_28_place/tl_2025_28_place.shp"
ms_gdf = gpd.read_file(ms_path)

print(ms_gdf.crs)
print(ms_gdf.columns)
ms_gdf.head()

ms_panel_long = build_places_panel_long(
    ms_gdf,
    districts_by_year,
    state_name="Mississippi",
    name_col_in="NAMELSAD",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(ms_panel_long)
n_split_rows = (ms_panel_long["is_split"] == 1).sum()
n_unified_rows = (ms_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
ms_panel_long.head()



EPSG:4269
Index(['STATEFP', 'PLACEFP', 'PLACENS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD',
       'LSAD', 'CLASSFP', 'PCICBSA', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER',
       'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 235 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 232 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 228 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 220 dropped geometries of

Total rows: 3416 | Split rows: 45 | Unified rows: 3371


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 144 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,2867200,Sherman town,1950,2,1,MS-01,0.747024,MS-02,0.252976,NaN,NaN,NaN,NaN,NaN,NaN,None
1,2815140,Collins city,1950,1,0,MS-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,2800300,Ackerman town,1950,1,0,MS-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,2850880,Nettleton city,1950,1,0,MS-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,2802700,Baldwyn city,1950,1,0,MS-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Alabama

al_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Alabama/Alabama_Incorporated_Cities_Boundaries_-1526713946241414309/Inc_Cities_Minus_Trib_01082020.shp"
al_gdf = gpd.read_file(al_path)

print(al_gdf.crs)
print(al_gdf.columns)
al_gdf.head()

dup_geoids = al_gdf["GEOID"][al_gdf["GEOID"].duplicated(keep=False)]
al_gdf = al_gdf[~al_gdf["GEOID"].isin(dup_geoids)].copy()

STATE_NAME = "Alabama"

districts_by_year_al = {}
for yr, g in districts_by_year.items():
    g2 = g[g["STATENAME"].astype(str).str.strip() == STATE_NAME].copy()
    districts_by_year_al[yr] = g2
    print(yr, len(g2))

districts_by_year_al = {
    yr: g[g["DISTRICT"] != 0].copy()
    for yr, g in districts_by_year_al.items()
}

al_panel_long = build_places_panel_long(
    al_gdf,
    districts_by_year_al,
    state_name="Alabama",
    name_col_in="NAME",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(al_panel_long)
n_split_rows = (al_panel_long["is_split"] == 1).sum()
n_unified_rows = (al_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
al_panel_long.head()


EPSG:4326
Index(['STATEFP', 'PLACEFP', 'PLACENS', 'AFFGEOID', 'GEOID', 'NAME', 'LSAD',
       'ALAND', 'AWATER', 'Locality_C', 'Locality_N', 'Name_1', 'County_Num',
       'TaxType', 'Rate_Type', 'Administer', 'Active_Dat', 'Inactive_D',
       'Rate', 'Indicator', 'PJ', 'County_Cod', 'PJ_Rate', 'Shape__Are',
       'Shape__Len', 'geometry'],
      dtype='object')
1950 10
1960 8
1970 7
1980 7
1990 7
2000 7
2010 7
2020 7
Total rows: 3656 | Split rows: 133 | Unified rows: 3523


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,0179344,Wadley,1950,1,0,AL-05,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,0100820,Alabaster,1950,1,0,AL-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,0129296,Gaylesville,1950,1,0,AL-05,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,0143816,Lockhart,1950,1,0,AL-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,0111512,Camden,1950,1,0,AL-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Tennessee

tn_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Tennessee/tl_2025_47_place/tl_2025_47_place.shp"
tn_gdf = gpd.read_file(tn_path)

print(tn_gdf.crs)
print(tn_gdf.columns)
print(tn_gdf.shape)
tn_gdf.head()

STATE_NAME = "Tennessee"

districts_by_year_tn = {}
for yr, g in districts_by_year.items():
    g2 = g[g["STATENAME"].astype(str).str.strip() == STATE_NAME].copy()
    districts_by_year_tn[yr] = g2
    print(yr, len(g2))

districts_by_year_tn = {
    yr: g[g["DISTRICT"] != 0].copy()
    for yr, g in districts_by_year_tn.items()
}

tn_panel_long = build_places_panel_long(
    tn_gdf,
    districts_by_year_tn,
    state_name="Tennessee",
    name_col_in="NAME",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(tn_panel_long)
n_split_rows = (tn_panel_long["is_split"] == 1).sum()
n_unified_rows = (tn_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
tn_panel_long.head()



EPSG:4269
Index(['STATEFP', 'PLACEFP', 'PLACENS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD',
       'LSAD', 'CLASSFP', 'PCICBSA', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER',
       'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')
(504, 17)
1950 9
1960 9
1970 8
1980 9
1990 9
2000 9
2010 9
2020 9


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 142 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 142 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 145 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 115 dropped geometries of

Total rows: 4032 | Split rows: 124 | Unified rows: 3908


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 574 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,4714000,Chattanooga,1950,1,0,TN-03,0.999988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,4716300,Collegedale,1950,1,0,TN-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,4769560,Soddy-Daisy,1950,1,0,TN-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,4743640,Lookout Mountain,1950,1,0,TN-03,0.999951,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,4721420,Dowelltown,1950,1,0,TN-04,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Missouri

mo_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Missouri/tl_2025_29_place/tl_2025_29_place.shp"
mo_gdf = gpd.read_file(mo_path)

print(mo_gdf.crs)
print(mo_gdf.columns)
print(mo_gdf.shape)
mo_gdf.head()

mo_panel_long = build_places_panel_long(
    mo_gdf,
    districts_by_year,
    state_name="Missouri",
    name_col_in="NAMELSAD",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(mo_panel_long)
n_split_rows = (mo_panel_long["is_split"] == 1).sum()
n_unified_rows = (mo_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
mo_panel_long.head()



EPSG:4269
Index(['STATEFP', 'PLACEFP', 'PLACENS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD',
       'LSAD', 'CLASSFP', 'PCICBSA', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER',
       'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')
(1081, 17)


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 121 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 144 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 158 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 73 dropped geometries of 

Total rows: 8648 | Split rows: 252 | Unified rows: 8396


/Users/mayarabin/Desktop/Thesis Files/.venv/lib/python3.8/site-packages/geopandas/geodataframe.py:1803: FutureWarning: `unary_union` returned None due to all-None GeoSeries. In future, `unary_union` will return 'GEOMETRYCOLLECTION EMPTY' instead.
  merged_geom = block.unary_union
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 482 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,2910450,Calhoun city,1950,1,0,MO-04,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,2956272,Park Hills city,1950,1,0,MO-08,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,2954038,Odessa city,1950,1,0,MO-04,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,2901054,Amity town,1950,1,0,MO-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,2978028,Weatherby town,1950,1,0,MO-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Georgia

ga_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Georgia/tl_2025_13_place/tl_2025_13_place.shp"
ga_gdf = gpd.read_file(ga_path)

print(ga_gdf.crs)
print(ga_gdf.columns)
print(ga_gdf.shape)
ga_gdf.head()

ga_panel_long = build_places_panel_long(
    ga_gdf,
    districts_by_year,
    state_name="Georgia",
    name_col_in="NAMELSAD",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(ga_panel_long)
n_split_rows = (ga_panel_long["is_split"] == 1).sum()
n_unified_rows = (ga_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
ga_panel_long.head()


EPSG:4269
Index(['STATEFP', 'PLACEFP', 'PLACENS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD',
       'LSAD', 'CLASSFP', 'PCICBSA', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER',
       'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')
(676, 17)


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 408 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 575 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 568 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 615 dropped geometries of

Total rows: 5408 | Split rows: 277 | Unified rows: 5131


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 1022 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,1360004,Pembroke city,1950,1,0,GA-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,1318000,Colquitt city,1950,1,0,GA-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,1337396,Hawkinsville city,1950,1,0,GA-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,1332916,Girard town,1950,1,0,GA-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,1379416,Vidette city,1950,1,0,GA-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# South Carolina

sc_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/South Carolina/Statewide_Municipal_Areas/GISTRANS.MUNICIPAL_AREAS.shp"
sc_gdf = gpd.read_file(sc_path)

print(sc_gdf.crs)
print(sc_gdf.columns)
print(sc_gdf.shape)
sc_gdf.head()

sc_cousub = gpd.read_file("/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/South Carolina/tl_2025_45_place/tl_2025_45_place.shp")
sc_cousub = sc_cousub.to_crs(sc_gdf.crs)

sc_gdf = sc_gdf.copy()
sc_cousub = sc_cousub.copy()

sc_gdf["geometry"] = sc_gdf.geometry.make_valid()
sc_cousub["geometry"] = sc_cousub.geometry.make_valid()

sc_gdf = sc_gdf.reset_index(drop=True).copy()
sc_gdf["SC_MUNI_ID"] = sc_gdf.index.astype(str)

ov = gpd.overlay(
    sc_gdf[["SC_MUNI_ID", "CITY_NAME", "geometry"]],
    sc_cousub[["GEOID", "NAME", "geometry"]],
    how="intersection"
)
ov["ov_area"] = ov.geometry.area

best = (
    ov.sort_values("ov_area", ascending=False)
      .groupby("SC_MUNI_ID", as_index=False)
      .first()[["SC_MUNI_ID", "GEOID", "NAME", "ov_area"]]
)

sc_gdf_with_geoid = sc_gdf.merge(best[["SC_MUNI_ID", "GEOID"]], on="SC_MUNI_ID", how="left")

print("Missing GEOIDs:", sc_gdf_with_geoid["GEOID"].isna().sum())
print("Rows:", len(sc_gdf_with_geoid), "Unique GEOIDs:", sc_gdf_with_geoid["GEOID"].nunique())

dup_geoids = sc_gdf_with_geoid["GEOID"][sc_gdf_with_geoid["GEOID"].duplicated(keep=False)]
sc_gdf_with_geoid = sc_gdf_with_geoid[~sc_gdf_with_geoid["GEOID"].isin(dup_geoids)].copy()

sc_panel_long = build_places_panel_long(
    sc_gdf_with_geoid,
    districts_by_year,
    state_name="South Carolina",
    name_col_in="CITY_NAME",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(sc_panel_long)
n_split_rows = (sc_panel_long["is_split"] == 1).sum()
n_unified_rows = (sc_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
sc_panel_long.head()



EPSG:32133
Index(['CITY_NAME', 'COUNTY_ID', 'COUNTY_NAM', 'Shape__Are', 'Shape__Len',
       'geometry'],
      dtype='object')
(272, 6)
Missing GEOIDs: 0
Rows: 272 Unique GEOIDs: 271
Total rows: 2160 | Split rows: 85 | Unified rows: 2075


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,4526845,FORT LAWN,1950,1,0,SC-05,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,4506040,BETHUNE,1950,1,0,SC-05,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,4523830,ESTILL,1950,1,0,SC-01,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,4541965,LITTLE MOUNTAIN,1950,1,0,SC-03,0.993692,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,4539310,LAKE CITY,1950,1,0,SC-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Kentucky

ky_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Kentucky/Corporate_Boundary_Polygons/Corporate_Boundary_Polygons.shp"
ky_gdf = gpd.read_file(ky_path)

print(ky_gdf.crs)
print(ky_gdf.columns)
print(ky_gdf.shape)

ky_gdf["STPLACEFP"] = ky_gdf["FIPS"].astype(str).str[:2] + ky_gdf["FIPS"].astype(str).str[-5:]
ky_gdf.head()

ky_gdf = ky_gdf[ky_gdf["STPLACEFP"].astype(str).str.len() >= 7].copy()
ky_gdf = ky_gdf[ky_gdf["NAME"] != "ELIZABETHTOWN"].copy()

ky_panel_long = build_places_panel_long(
    ky_gdf,
    districts_by_year,
    state_name="Kentucky",
    name_col_in="NAME",
    county_col_in=None,
    id_source_col_in="STPLACEFP",
    THRESH=0.03
)

n_rows = len(ky_panel_long)
n_split_rows = (ky_panel_long["is_split"] == 1).sum()
n_unified_rows = (ky_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
ky_panel_long.head()


EPSG:3857
Index(['OBJECTID', 'NAME', 'FIPS', 'COUNTY', 'ELEV', 'COSEAT', 'INCORP',
       'CLASS', 'CORP_AREA', 'CONTACT', 'LAST_UPDT', 'XY_SOURCE', 'XY_ISSUES',
       'COMMENTS', 'ADDNAME', 'SOSLINK', 'NAME2', 'AGENCY_CON', 'Full_SOS_L',
       'POP90', 'POP2000', 'POP2010', 'POPEST2011', 'POPEST2012', 'POPEST2013',
       'POPEST2014', 'POPEST2015', 'POPEST2016', 'CITYFIPS', 'Area_SqMil',
       'Shapearea', 'Shapelen', 'geometry'],
      dtype='object')
(426, 33)
Total rows: 3328 | Split rows: 55 | Unified rows: 3273


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,2100298,ADAIRVILLE,1950,1,0,KY-02,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,2100694,ALBANY,1950,1,0,KY-08,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,2100802,ALEXANDRIA,1950,1,0,KY-05,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,2100946,ALLEN,1950,1,0,KY-07,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,2101504,ANCHORAGE,1950,1,0,KY-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# New York

ny_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/New York/NYS_Civil_Boundaries_2137829920944419587/Cities_Towns.shp"
ny_gdf = gpd.read_file(ny_path)

print(ny_gdf.crs)
print(ny_gdf.columns)
ny_gdf.head()

ny_panel_long = build_places_panel_long(
    ny_gdf,
    districts_by_year,
    state_name="New York",
    name_col_in="NAME",
    county_col_in=None,
    id_source_col_in="FIPS_CODE",
    THRESH=0.03
)

n_rows = len(ny_panel_long)
n_split_rows = (ny_panel_long["is_split"] == 1).sum()
n_unified_rows = (ny_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
ny_panel_long.head()



EPSG:3857
Index(['NAME', 'MUNI_TYPE', 'MUNITYCODE', 'COUNTY', 'GNIS_ID', 'FIPS_CODE',
       'SWIS', 'POP1990', 'POP2000', 'POP2010', 'POP2020', 'DOS_LL',
       'DOSLL_DATE', 'MAP_SYMBOL', 'CALC_SQ_MI', 'DATEMOD', 'geometry'],
      dtype='object')
Total rows: 7960 | Split rows: 129 | Unified rows: 7831


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,3604500210,Adams,1950,1,0,NY-33,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,3610100287,Addison,1950,1,0,NY-37,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,3601700353,Afton,1950,1,0,NY-36,0.999144,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,3603700474,Alabama,1950,1,0,NY-39,0.99873,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,3600101000,Albany,1950,1,0,NY-30,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Virginia

va_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Virginia/tl_2025_51_cousub/tl_2025_51_cousub.shp"
va_gdf = gpd.read_file(va_path)

print(va_gdf.crs)
print(va_gdf.columns)
va_gdf.head()

va_panel_long = build_places_panel_long(
    va_gdf,
    districts_by_year,
    state_name="Virginia",
    name_col_in="NAME",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(va_panel_long)
n_split_rows = (va_panel_long["is_split"] == 1).sum()
n_unified_rows = (va_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
va_panel_long.head()


EPSG:4269
Index(['STATEFP', 'COUNTYFP', 'COUSUBFP', 'COUSUBNS', 'GEOID', 'GEOIDFQ',
       'NAME', 'NAMELSAD', 'LSAD', 'CLASSFP', 'MTFCC', 'FUNCSTAT', 'ALAND',
       'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 5242 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 5242 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 3954 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 54 dropped geometries 

Total rows: 4416 | Split rows: 167 | Unified rows: 4249


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,5173595115,Poquoson,1950,1,0,VA-01,0.206444,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,5104591032,Craig Creek,1950,1,0,VA-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,5108993919,Horsepasture,1950,1,0,VA-05,0.999957,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,5108995295,Ridgeway,1950,1,0,VA-05,0.999246,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,5108990288,Blackberry,1950,1,0,VA-05,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Ohio

oh_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Ohio/tl_2025_39_place/tl_2025_39_place.shp"
oh_gdf = gpd.read_file(oh_path)

print(oh_gdf.crs)
print(oh_gdf.columns)
oh_gdf.head()

oh_panel_long = build_places_panel_long(
    oh_gdf,
    districts_by_year,
    state_name="Ohio",
    name_col_in="NAMELSAD",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(oh_panel_long)
n_split_rows = (oh_panel_long["is_split"] == 1).sum()
n_unified_rows = (oh_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
oh_panel_long.head()



EPSG:4269
Index(['STATEFP', 'PLACEFP', 'PLACENS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD',
       'LSAD', 'CLASSFP', 'PCICBSA', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER',
       'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 271 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 271 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 168 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 222 dropped geometries of

Total rows: 10120 | Split rows: 2906 | Unified rows: 7214


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 1519 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,3946844,Magnolia village,1950,3,1,OH-01,1.0,OH-16,0.709105,OH-18,0.290895,NaN,NaN,NaN,NaN,None
1,3950834,Minerva village,1950,3,1,OH-01,1.0,OH-16,0.501322,OH-18,0.498678,NaN,NaN,NaN,NaN,None
2,3904458,Beach City village,1950,2,1,OH-01,1.0,OH-16,1.0,NaN,NaN,NaN,NaN,NaN,NaN,None
3,3906908,Blanchester village,1950,2,1,OH-01,1.0,OH-07,1.0,NaN,NaN,NaN,NaN,NaN,NaN,None
4,3915406,Clarksville village,1950,2,1,OH-01,1.0,OH-07,1.0,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Iowa

ia_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Iowa/tl_2025_19_place/tl_2025_19_place.shp"
ia_gdf = gpd.read_file(ia_path)

print(ia_gdf.crs)
print(ia_gdf.columns)
ia_gdf.head()

ia_panel_long = build_places_panel_long(
    ia_gdf,
    districts_by_year,
    state_name="Iowa",
    name_col_in="NAMELSAD",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(ia_panel_long)
n_split_rows = (ia_panel_long["is_split"] == 1).sum()
n_unified_rows = (ia_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
ia_panel_long.head()



EPSG:4269
Index(['STATEFP', 'PLACEFP', 'PLACENS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD',
       'LSAD', 'CLASSFP', 'PCICBSA', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER',
       'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 12 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 7 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 9 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 1 dropped geometries of differ

Total rows: 8208 | Split rows: 108 | Unified rows: 8100


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,1957630,Northwood city,1950,1,0,IA-03,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,1921360,Dickens city,1950,1,0,IA-08,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,1930810,Gillett Grove city,1950,1,0,IA-08,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,1968925,Rossie city,1950,1,0,IA-08,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,1931980,Grand Junction city,1950,1,0,IA-06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# Illinois

il_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Place_GIS_STATE/Illinois/tl_2025_17_place/tl_2025_17_place.shp"
il_gdf = gpd.read_file(il_path)

print(il_gdf.crs)
print(il_gdf.columns)
il_gdf.head()

il_panel_long = build_places_panel_long(
    il_gdf,
    districts_by_year,
    state_name="Illinois",
    name_col_in="NAMELSAD",
    county_col_in=None,
    id_source_col_in="GEOID",
    THRESH=0.03
)

n_rows = len(il_panel_long)
n_split_rows = (il_panel_long["is_split"] == 1).sum()
n_unified_rows = (il_panel_long["is_split"] == 0).sum()
print(f"Total rows: {n_rows} | Split rows: {n_split_rows} | Unified rows: {n_unified_rows}")
il_panel_long.head()


EPSG:4269
Index(['STATEFP', 'PLACEFP', 'PLACENS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD',
       'LSAD', 'CLASSFP', 'PCICBSA', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER',
       'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 11 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 27 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 22 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(
/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 107 dropped geometries of di

Total rows: 11688 | Split rows: 893 | Unified rows: 10795


/var/folders/v1/kq7_n16n0rs5r7zt94_0r2rh0000gn/T/ipykernel_3852/84423743.py:69: UserWarning: `keep_geom_type=True` in overlay resulted in 2499 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  pieces = gpd.overlay(


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,1710110,Bushnell city,1950,1,0,IL-20,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,1715378,Colchester city,1950,1,0,IL-20,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,1737439,Industry village,1950,1,0,IL-20,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,1761548,Prairie City village,1950,1,0,IL-20,0.992537,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,1768198,Sciota village,1950,1,0,IL-20,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:

# --- Build places_panel: Continental U.S. only (exclude AK, HI) ---

state_panels = [
    # New England
    vt_panel_long, nh_panel_long, me_panel_long, ma_panel_long,
    ri_panel_long, ct_panel_long,
    # Mid-Atlantic
    ny_panel_long, nj_panel_long, pa_panel_long, de_panel_long, md_panel_long,
    # South Atlantic
    va_panel_long, wv_panel_long, nc_panel_long, sc_panel_long,
    ga_panel_long, fl_panel_long,
    # East South Central
    ky_panel_long, tn_panel_long, al_panel_long, ms_panel_long,
    # Midwest
    mi_panel_long, oh_panel_long, in_panel_long, il_panel_long,
    wi_panel_long, mn_panel_long,
    # Plains/Central US
    ia_panel_long, ks_panel_long, mo_panel_long, ne_panel_long,
    nd_panel_long, sd_panel_long,
    # West South Central
    ar_panel_long, la_panel_long, ok_panel_long, tx_panel_long,
    # Corner States
    co_panel_long, ut_panel_long, az_panel_long, nm_panel_long, nv_panel_long,
    # Mountain
    id_panel_long, mt_panel_long, wy_panel_long,
    # Pacific
    ca_panel_long, or_panel_long, wa_panel_long,
]

all_states_panel = pd.concat(state_panels, ignore_index=True)


# %%
print("Total rows:", len(all_states_panel))
print("Unique PLACE_ID:", all_states_panel["PLACE_ID"].nunique())
print("Years:", sorted(all_states_panel["year"].unique()))
print("Split count:", all_states_panel["is_split"].sum())
all_states_panel.head()



Total rows: 244028
Unique PLACE_ID: 30855
Years: [1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020]
Split count: 11362


,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY
0,5000911800,Canaan,1950,1,0,VT-01,0.99678,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,5001127100,Franklin,1950,1,0,VT-01,0.998669,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,5001105425,Berkshire,1950,1,0,VT-01,0.999732,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,5001133025,Highgate,1950,1,0,VT-01,0.99754,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,5001159125,Richford,1950,1,0,VT-01,0.999641,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
# ============================================================
# PLACE_ID CLEANUP (unchanged from original)
# ============================================================

VALID_STATEFPS = {
    "01","02","04","05","06","08","09","10","11","12","13","15","16","17","18","19","20","21","22","23","24","25",
    "26","27","28","29","30","31","32","33","34","35","36","37","38","39","40","41","42","44","45","46","47","48",
    "49","50","51","53","54","55","56"
}

def drop_invalid_statefp_rows(places_panel: pd.DataFrame) -> pd.DataFrame:
    cleaned = places_panel.copy()
    place_id = cleaned["PLACE_ID"].astype(str).str.strip()
    statefp = place_id.str.slice(0, 2)
    valid = statefp.isin(VALID_STATEFPS)
    before = len(cleaned)
    cleaned = cleaned.loc[valid].copy()
    after = len(cleaned)
    print(f"Dropped {before - after:,} rows with invalid statefp (including '00').")
    print("Remaining statefp counts (top 10):")
    print(statefp.loc[valid].value_counts().head(10))
    return cleaned


def fix_place_id_len6(places_panel: pd.DataFrame, id_col="PLACE_ID") -> pd.DataFrame:
    places_panel = places_panel.copy()
    places_panel[id_col] = places_panel[id_col].astype(str).str.strip()
    mask_len6 = places_panel[id_col].str.len() == 6
    if mask_len6.any():
        statefp = places_panel.loc[mask_len6, id_col].str.slice(0, 2)
        tail4   = places_panel.loc[mask_len6, id_col].str.slice(2)
        places_panel.loc[mask_len6, id_col] = statefp + tail4.str.zfill(5)
    return places_panel


def clean_place_id_lengths(places_panel: pd.DataFrame) -> pd.DataFrame:
    cleaned = places_panel.copy()
    cleaned["PLACE_ID_str"] = cleaned["PLACE_ID"].astype(str).str.strip()
    junk = {"", "nan", "NaN", "None", "<NA>"}
    cleaned.loc[cleaned["PLACE_ID_str"].isin(junk), "PLACE_ID_str"] = pd.NA
    before = len(cleaned)
    cleaned = cleaned.dropna(subset=["PLACE_ID_str"])
    cleaned = cleaned[cleaned["PLACE_ID_str"].str.len() > 1].copy()
    cleaned["PLACE_ID"] = cleaned["PLACE_ID_str"]
    cleaned = cleaned.drop(columns=["PLACE_ID_str"])
    after = len(cleaned)
    print(f"Dropped {before - after:,} rows with missing/length<=1 PLACE_ID")
    lengths = cleaned["PLACE_ID"].astype(str).str.len().value_counts().sort_index()
    print("Remaining PLACE_ID length counts:\n", lengths)
    return cleaned


all_states_panel = clean_place_id_lengths(all_states_panel)
all_states_panel = drop_invalid_statefp_rows(all_states_panel)
all_states_panel = fix_place_id_len6(all_states_panel)

all_states_panel = all_states_panel[
    all_states_panel["PLACE_ID"].astype(str).str.len().isin([7, 10])
].copy()

all_states_panel = fix_place_id_len6(all_states_panel, id_col="PLACE_ID")

print("PLACE_ID length counts after len-6 fix:\n",
      all_states_panel["PLACE_ID"].str.len().value_counts().sort_index())


Dropped 8 rows with missing/length<=1 PLACE_ID
Remaining PLACE_ID length counts:
 PLACE_ID
6         16
7     136148
10    107856
Name: count, dtype: int64
Dropped 8 rows with invalid statefp (including '00').
Remaining statefp counts (top 10):
PLACE_ID
27    21472
42    20552
55    15304
20    12168
26    12168
17    11688
39    10120
48     9792
29     8648
19     8208
Name: count, dtype: int64
PLACE_ID length counts after len-6 fix:
 PLACE_ID
7     136156
10    107856
Name: count, dtype: int64


In [ ]:
# ============================================================
# CHANGE 3: Attach Cook PVI to all district slots, then save
#
# cook_year is set per panel decade using PANEL_YEAR_TO_COOK_YEAR.
# For each of the dist1..dist5 columns produced by the overlay,
# we look up (cook_year, dist_code) in cook_lookup and attach
# pvi_score, pvi_party, and pvi_string.
# Pre-1990 rows get NaN for all PVI columns — correct by design.
# ============================================================

all_states_panel["cook_year"] = all_states_panel["year"].map(PANEL_YEAR_TO_COOK_YEAR)

for i in range(1, 6):
    dist_col = f"dist{i}"
    if dist_col not in all_states_panel.columns:
        continue

    left = all_states_panel[["cook_year", dist_col]].copy()
    left.columns = ["cook_year", "dist_code"]

    merged = left.merge(cook_lookup, on=["cook_year", "dist_code"], how="left")

    all_states_panel[f"dist{i}_pvi"]        = merged["pvi_score"].values
    all_states_panel[f"dist{i}_pvi_party"]  = merged["pvi_party"].values
    all_states_panel[f"dist{i}_pvi_string"] = merged["pvi_string"].values

# %%
# Sanity check
print("Final panel shape:", all_states_panel.shape)
print("Columns:", all_states_panel.columns.tolist())

print("\nPVI coverage by decade:")
for yr in sorted(all_states_panel["year"].unique()):
    sub = all_states_panel[all_states_panel["year"] == yr]
    has_pvi = sub["dist1_pvi"].notna().sum()
    print(f"  {yr}: {has_pvi:,} / {len(sub):,} places have dist1 PVI")

print("\nSample split city with PVI:")
print(
    all_states_panel[all_states_panel["is_split"] == 1]
    [[
        "PLACE_ID", "NAME", "year", "n_districts",
        "dist1", "dist1_share", "dist1_pvi", "dist1_pvi_party",
        "dist2", "dist2_share", "dist2_pvi", "dist2_pvi_party",
    ]]
    .dropna(subset=["dist1_pvi"])
    .head(10)
)

# %%
# Save — definitive panel, never need to rerun again
all_states_panel.to_parquet("places_panel_with_districts.parquet", index=False)
print("Saved to places_panel_with_districts.parquet")



Final panel shape: (244012, 32)
Columns: ['PLACE_ID', 'NAME', 'year', 'n_districts', 'is_split', 'dist1', 'dist1_share', 'dist2', 'dist2_share', 'dist3', 'dist3_share', 'dist4', 'dist4_share', 'dist5', 'dist5_share', 'COUNTY', 'cook_year', 'dist1_pvi', 'dist1_pvi_party', 'dist1_pvi_string', 'dist2_pvi', 'dist2_pvi_party', 'dist2_pvi_string', 'dist3_pvi', 'dist3_pvi_party', 'dist3_pvi_string', 'dist4_pvi', 'dist4_pvi_party', 'dist4_pvi_string', 'dist5_pvi', 'dist5_pvi_party', 'dist5_pvi_string']

PVI coverage by decade:
  1950: 0 / 30,853 places have dist1 PVI
  1960: 0 / 30,853 places have dist1 PVI
  1970: 0 / 30,853 places have dist1 PVI
  1980: 0 / 30,368 places have dist1 PVI
  1990: 30,234 / 30,239 places have dist1 PVI
  2000: 30,235 / 30,239 places have dist1 PVI
  2010: 29,930 / 30,239 places have dist1 PVI
  2020: 30,142 / 30,368 places have dist1 PVI

Sample split city with PVI:
         PLACE_ID        NAME  year  n_districts  dist1 dist1_share  \
9758   2502737420   LUNENBU

In [ ]:

# %%
# ============================================================
# ACS CONTROLS PANEL (unchanged from original)
# ============================================================

API_KEY = "c30a039e986960847fce056581e7294e88867620"
ACS_YEARS = [2012, 2017, 2022]

ACS_CORE_VARS = [
    "NAME",
    "B01003_001E", "B01002_001E",
    "B02001_002E", "B02001_003E", "B02001_005E", "B03003_003E",
    "B15003_001E", "B15003_022E", "B15003_023E", "B15003_024E", "B15003_025E",
    "B19013_001E", "B19083_001E",
    "B17001_001E", "B17001_002E",
    "B23025_003E", "B23025_005E",
    "B25003_001E", "B25003_002E",
    "B25077_001E", "B25064_001E",
]

ACS_AGE_DETAIL_VARS = [
    "B01001_001E",
    "B01001_003E","B01001_004E","B01001_005E","B01001_006E","B01001_007E",
    "B01001_008E","B01001_009E","B01001_010E","B01001_011E","B01001_012E","B01001_013E",
    "B01001_014E","B01001_015E","B01001_016E","B01001_017E",
    "B01001_018E","B01001_019E","B01001_020E","B01001_021E","B01001_022E","B01001_023E",
    "B01001_024E","B01001_025E",
    "B01001_027E","B01001_028E","B01001_029E","B01001_030E","B01001_031E",
    "B01001_032E","B01001_033E","B01001_034E","B01001_035E","B01001_036E","B01001_037E",
    "B01001_038E","B01001_039E","B01001_040E","B01001_041E",
    "B01001_042E","B01001_043E","B01001_044E","B01001_045E","B01001_046E","B01001_047E",
    "B01001_048E","B01001_049E",
]

ACS_ALL_VARS = ACS_CORE_VARS + ACS_AGE_DETAIL_VARS


def chunk_list(items, chunk_size=40):
    return [items[i:i+chunk_size] for i in range(0, len(items), chunk_size)]


def infer_state_id_length_map(places_panel: pd.DataFrame) -> dict:
    places_panel = places_panel.copy()
    places_panel["PLACE_ID_str"] = places_panel["PLACE_ID"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    places_panel["statefp"] = places_panel["PLACE_ID_str"].str[:2]
    places_panel["id_len"] = places_panel["PLACE_ID_str"].str.len()
    id_len_map = (
        places_panel.groupby("statefp")["id_len"]
        .agg(lambda s: s.value_counts().idxmax())
        .to_dict()
    )
    bad = {k: v for k, v in id_len_map.items() if v not in (7, 10)}
    if bad:
        raise ValueError(f"Found states with unexpected PLACE_ID lengths (expected 7 or 10): {bad}")
    return id_len_map


def pull_state_acs5_raw(year: int, statefp_2: str, id_length: int, api_key: str, variables: list, chunk_size: int = 40) -> pd.DataFrame:
    if year < 2009:
        raise ValueError(f"ACS5 not available for {year}.")

    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=2,
        status_forcelist=[500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False
    )
    session.mount("https://", HTTPAdapter(max_retries=retries))

    url = f"https://api.census.gov/data/{year}/acs/acs5"
    var_chunks = chunk_list(variables, chunk_size=chunk_size)

    frames = []
    for vars_chunk in var_chunks:
        params = {
            "get": ",".join(vars_chunk),
            "in": f"state:{statefp_2}",
            "key": api_key
        }
        if id_length == 7:
            params["for"] = "place:*"
        elif id_length == 10:
            params["for"] = "county subdivision:*"
            params["in"] = f"state:{statefp_2} county:*"
        else:
            raise ValueError("id_length must be 7 or 10")

        # Retry loop for read timeouts specifically
        for attempt in range(5):
            try:
                r = session.get(url, params=params, timeout=(10, 120))  # (connect timeout, read timeout)
                r.raise_for_status()
                break
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
                wait = 2 ** attempt
                print(f"  Attempt {attempt+1} failed ({e.__class__.__name__}), retrying in {wait}s...")
                time.sleep(wait)
        else:
            raise RuntimeError(f"Failed to pull state={statefp_2} year={year} after 5 attempts")

        data = r.json()
        frames.append(pd.DataFrame(data[1:], columns=data[0]))

    key_cols = ["state"] + (["place"] if id_length == 7 else ["county", "county subdivision"])
    merged = frames[0]
    for f in frames[1:]:
        overlap = (set(merged.columns) & set(f.columns)) - set(key_cols)
        merged = merged.merge(f.drop(columns=list(overlap), errors="ignore"), on=key_cols, how="outer")

    merged["year"] = year
    return merged


In [ ]:
def safe_log1p(series: pd.Series) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce")
    return np.log1p(x.where(x >= 0))


def postprocess_acs_to_place_id(acs_raw: pd.DataFrame, id_length: int) -> pd.DataFrame:
    df = acs_raw.copy()

    df["state"] = df["state"].astype(str).str.zfill(2)
    if id_length == 7:
        df["place"] = df["place"].astype(str).str.zfill(5)
        df["PLACE_ID"] = df["state"] + df["place"]
    else:
        df["county"] = df["county"].astype(str).str.zfill(3)
        df["county subdivision"] = df["county subdivision"].astype(str).str.zfill(5)
        df["PLACE_ID"] = df["state"] + df["county"] + df["county subdivision"]

    for c in df.columns:
        if c.startswith("B") and c.endswith("E"):
            df[c] = pd.to_numeric(df[c], errors="coerce")

    rename_map = {
        "NAME": "area_name",
        "B01003_001E": "pop_total", "B01002_001E": "age_median",
        "B02001_002E": "pop_white", "B02001_003E": "pop_black",
        "B02001_005E": "pop_asian", "B03003_003E": "pop_hispanic",
        "B15003_001E": "pop_25plus",
        "B15003_022E": "edu_ba", "B15003_023E": "edu_ma",
        "B15003_024E": "edu_prof", "B15003_025E": "edu_phd",
        "B19013_001E": "hh_income_median", "B19083_001E": "gini",
        "B17001_001E": "pov_universe", "B17001_002E": "pov_below",
        "B23025_003E": "labor_force", "B23025_005E": "unemployed",
        "B25003_001E": "housing_occupied", "B25003_002E": "housing_owner_occ",
        "B25077_001E": "home_value_median", "B25064_001E": "rent_median",
    }
    df = df.rename(columns=rename_map)

    def row_sum(cols):
        return df[cols].sum(axis=1, skipna=True)

    cols_u20 = ["B01001_003E","B01001_004E","B01001_005E","B01001_006E","B01001_007E",
                "B01001_027E","B01001_028E","B01001_029E","B01001_030E","B01001_031E"]
    cols_20_39 = ["B01001_008E","B01001_009E","B01001_010E","B01001_011E","B01001_012E","B01001_013E",
                  "B01001_032E","B01001_033E","B01001_034E","B01001_035E","B01001_036E","B01001_037E"]
    cols_40_59 = ["B01001_014E","B01001_015E","B01001_016E","B01001_017E",
                  "B01001_038E","B01001_039E","B01001_040E","B01001_041E"]
    cols_60_79 = ["B01001_018E","B01001_019E","B01001_020E","B01001_021E","B01001_022E","B01001_023E",
                  "B01001_042E","B01001_043E","B01001_044E","B01001_045E","B01001_046E","B01001_047E"]
    cols_80p = ["B01001_024E","B01001_025E","B01001_048E","B01001_049E"]

    pop = df["pop_total"].replace({0: np.nan})
    df["age_share_u20"]   = row_sum(cols_u20) / pop
    df["age_share_20_39"] = row_sum(cols_20_39) / pop
    df["age_share_40_59"] = row_sum(cols_40_59) / pop
    df["age_share_60_79"] = row_sum(cols_60_79) / pop
    df["age_share_80p"]   = row_sum(cols_80p) / pop
    df["age_share_sum_check"] = (df["age_share_u20"] + df["age_share_20_39"] +
                                  df["age_share_40_59"] + df["age_share_60_79"] + df["age_share_80p"])

    df["log_pop"]        = safe_log1p(df["pop_total"])
    df["log_mhi"]        = safe_log1p(df["hh_income_median"])
    df["share_black"]    = df["pop_black"] / pop
    df["share_asian"]    = df["pop_asian"] / pop
    df["share_hispanic"] = df["pop_hispanic"] / pop
    df["share_ba_plus"]  = (df["edu_ba"] + df["edu_ma"] + df["edu_prof"] + df["edu_phd"]) / df["pop_25plus"].replace({0: np.nan})
    df["poverty_rate"]   = df["pov_below"] / df["pov_universe"].replace({0: np.nan})
    df["unemp_rate"]     = df["unemployed"] / df["labor_force"].replace({0: np.nan})
    df["homeown_rate"]   = df["housing_owner_occ"] / df["housing_occupied"].replace({0: np.nan})

    drop_geo_cols    = ["state", "place", "county", "county subdivision", "NAME"]
    drop_b01001_bins = [c for c in df.columns if c.startswith("B01001_")]
    df = df.drop(columns=drop_geo_cols + drop_b01001_bins, errors="ignore")

    front = ["PLACE_ID", "year", "area_name"]
    front = [c for c in front if c in df.columns]
    df = df[front + [c for c in df.columns if c not in front]]

    return df


In [ ]:
def build_acs_controls_panel(places_panel: pd.DataFrame, years: List[int], api_key: str) -> pd.DataFrame:
    id_len_map = infer_state_id_length_map(places_panel)
    states_in_panel = sorted(id_len_map.keys())

    frames = []
    for yr in years:
        for statefp in states_in_panel:
            id_len = int(id_len_map[statefp])
            print(f"Pulling ACS{yr} state={statefp} id_len={id_len} ...")
            raw = pull_state_acs5_raw(
                year=yr, statefp_2=statefp, id_length=id_len,
                api_key=api_key, variables=ACS_ALL_VARS, chunk_size=40
            )
            tidy = postprocess_acs_to_place_id(raw, id_len)
            frames.append(tidy)

    return pd.concat(frames, ignore_index=True)


# %%
acs_controls_panel = build_acs_controls_panel(
    places_panel=all_states_panel,
    years=ACS_YEARS,
    api_key=API_KEY
)

print("acs_controls_panel shape:", acs_controls_panel.shape)
acs_controls_panel.head()

# %%
acs_controls_panel.to_parquet("acs_controls_panel.parquet")

Pulling ACS2012 state=01 id_len=7 ...
  Attempt 1 failed (ConnectionError), retrying in 1s...
  Attempt 2 failed (ConnectTimeout), retrying in 2s...
Pulling ACS2012 state=04 id_len=7 ...
Pulling ACS2012 state=05 id_len=7 ...
Pulling ACS2012 state=06 id_len=7 ...
Pulling ACS2012 state=08 id_len=7 ...
Pulling ACS2012 state=09 id_len=10 ...
Pulling ACS2012 state=10 id_len=10 ...
Pulling ACS2012 state=12 id_len=7 ...
Pulling ACS2012 state=13 id_len=7 ...
Pulling ACS2012 state=16 id_len=7 ...
Pulling ACS2012 state=17 id_len=7 ...
Pulling ACS2012 state=18 id_len=10 ...
Pulling ACS2012 state=19 id_len=7 ...
Pulling ACS2012 state=20 id_len=10 ...
Pulling ACS2012 state=21 id_len=7 ...
Pulling ACS2012 state=22 id_len=7 ...
Pulling ACS2012 state=23 id_len=10 ...
Pulling ACS2012 state=24 id_len=10 ...
Pulling ACS2012 state=25 id_len=10 ...
Pulling ACS2012 state=26 id_len=7 ...
Pulling ACS2012 state=27 id_len=10 ...
Pulling ACS2012 state=28 id_len=7 ...
Pulling ACS2012 state=29 id_len=7 ...
Pulling

In [ ]:
# # %%
# # combines all of the data on census districts from 1950 to 2020

# dfs = [gdf_1950s_Dist, gdf_1960s_Dist, gdf_1970s_Dist, gdf_1980s_Dist,
#        gdf_1990s_Dist, gdf_2000s_Dist, gdf_2010s_Dist, gdf_2020s_Dist]
# Districts = pd.concat(dfs, ignore_index=True)

# gdf_1950s_Dist = gdf_1950s_Dist.to_crs(5070)
# gdf_1960s_Dist = gdf_1960s_Dist.to_crs(5070)
# gdf_1970s_Dist = gdf_1970s_Dist.to_crs(5070)
# gdf_1980s_Dist = gdf_1980s_Dist.to_crs(5070)
# gdf_1990s_Dist = gdf_1990s_Dist.to_crs(5070)
# gdf_2000s_Dist = gdf_2000s_Dist.to_crs(5070)
# gdf_2010s_Dist = gdf_2010s_Dist.to_crs(5070)
# gdf_2020s_Dist = gdf_2020s_Dist.to_crs(5070)


In [ ]:
Districts.head()

,STATENAME,DISTRICT,STARTCONG,ENDCONG,ID,DISTRICTSI,COUNTY,PAGE,LAW,NOTE,BESTDEC,FINALNOTE,RNOTE,LASTCHANGE,FROMCOUNTY,STATEFP,geometry,split_year
0,Alabama,0.0,16.0,88.0,001016088000,"{1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1};{260,260,260,2...","{All,All,All,All,All,All,All,All,All,All,All,All,All,All,All,All,All,All,All...","{218,218,218,218,218,218,218,218,218,218,218,218,218,218,218,218,218,218,218...","{""Congressional District Law January 1, 1841 (N. 46)"",""Congressional Distric...","{""Eight general ticket representatives"",""Eight general ticket representative...","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,...","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,...","{""Eight general ticket representatives"",""Eight general ticket representative...",2011-04-05,T,None,"MULTIPOLYGON (((-88.05205 30.19698, -88.05337 30.19537, -88.05419 30.19469, ...",1950
1,Alabama,1.0,73.0,87.0,001073087001,"{731,728,729,727,732,733,730}","{Monroe,Clarke,Marengo,Choctaw,Washington,Wilcox,Mobile}","{218,218,218,218,218,218,218}","{NULL,NULL,NULL,""Congressional District Law February 25, 1931 (N. 59)"",NULL,...","{NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL}",2011-04-05,T,None,"MULTIPOLYGON (((-88.05205 30.19698, -88.05337 30.19537, -88.05419 30.19469, ...",1950
2,Alabama,2.0,73.0,87.0,001073087002,"{736,734,735,738,741,742,740,739,737}","{Conecuh,Baldwin,Butler,Crenshaw,Montgomery,Pike,Lowndes,Escambia,Covington}","{218,218,218,218,218,218,218,218,218}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}",2011-04-05,T,None,"MULTIPOLYGON (((-88.00584 30.74350, -88.00625 30.74341, -88.00660 30.74339, ...",1950
3,Alabama,3.0,73.0,87.0,001073087003,"{752,751,746,743,747,748,749,744,745,750}","{Russell,Macon,Dale,Barbour,Geneva,Henry,Houston,Bullock,Coffee,Lee}","{218,218,218,218,218,218,218,218,218,218}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}",2011-04-05,T,None,"POLYGON ((-85.47862 30.99720, -85.48830 30.99706, -85.49263 30.99700, -85.49...",1950
4,Alabama,4.0,73.0,87.0,001073087004,"{760,757,759,758,755,753,756,754}","{Talladega,Dallas,""St. Clair"",Elmore,Clay,Autauga,Coosa,Calhoun}","{218,218,218,218,218,218,218,218}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}",2011-04-05,T,None,"POLYGON ((-87.01753 32.67771, -87.01760 32.66984, -87.01766 32.66327, -87.00...",1950


In [ ]:
# creates the fucntion and calculates the reock score for each congressional district


def min_bounding_circle_area(hull):
    """
    Approximate minimum bounding circle using the convex hull vertices.
    Finds the two points that are furthest apart (the diameter) and uses
    that as the bounding circle diameter.
    """
    coords = np.array(hull.exterior.coords)
    max_dist_sq = 0
    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):
            diff = coords[i] - coords[j]
            dist_sq = diff[0]**2 + diff[1]**2
            if dist_sq > max_dist_sq:
                max_dist_sq = dist_sq
    radius = np.sqrt(max_dist_sq) / 2
    return np.pi * radius**2

def reock_score(geometry):
    if geometry is None or geometry.is_empty:
        return np.nan
    
    hull = geometry.convex_hull
    
    # Handle edge case where hull is a point or line
    if hull.geom_type not in ['Polygon', 'MultiPolygon']:
        return np.nan
    
    mbc_area = min_bounding_circle_area(hull)
    
    if mbc_area == 0:
        return np.nan
    
    return geometry.area / mbc_area

print("Calculating Reock scores...")
Districts['reock'] = Districts['geometry'].apply(reock_score)

print(f"Done. Shape: {Districts.shape}")
print(f"Missing values: {Districts['reock'].isna().sum()}")
print(Districts['reock'].describe())

Calculating Reock scores...
Done. Shape: (3502, 19)
Missing values: 3
count    3499.000000
mean        0.364018
std         0.124378
min         0.000063
25%         0.278433
50%         0.363717
75%         0.453023
max         0.773438
Name: reock, dtype: float64


In [ ]:
Districts.head()

,STATENAME,DISTRICT,STARTCONG,ENDCONG,ID,DISTRICTSI,COUNTY,PAGE,LAW,NOTE,BESTDEC,FINALNOTE,RNOTE,LASTCHANGE,FROMCOUNTY,STATEFP,geometry,split_year,reock
0,Alabama,0.0,16.0,88.0,001016088000,"{1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1};{260,260,260,2...","{All,All,All,All,All,All,All,All,All,All,All,All,All,All,All,All,All,All,All...","{218,218,218,218,218,218,218,218,218,218,218,218,218,218,218,218,218,218,218...","{""Congressional District Law January 1, 1841 (N. 46)"",""Congressional Distric...","{""Eight general ticket representatives"",""Eight general ticket representative...","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,...","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,...","{""Eight general ticket representatives"",""Eight general ticket representative...",2011-04-05,T,None,"MULTIPOLYGON (((-88.05205 30.19698, -88.05337 30.19537, -88.05419 30.19469, ...",1950,0.547522
1,Alabama,1.0,73.0,87.0,001073087001,"{731,728,729,727,732,733,730}","{Monroe,Clarke,Marengo,Choctaw,Washington,Wilcox,Mobile}","{218,218,218,218,218,218,218}","{NULL,NULL,NULL,""Congressional District Law February 25, 1931 (N. 59)"",NULL,...","{NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL}",2011-04-05,T,None,"MULTIPOLYGON (((-88.05205 30.19698, -88.05337 30.19537, -88.05419 30.19469, ...",1950,0.409546
2,Alabama,2.0,73.0,87.0,001073087002,"{736,734,735,738,741,742,740,739,737}","{Conecuh,Baldwin,Butler,Crenshaw,Montgomery,Pike,Lowndes,Escambia,Covington}","{218,218,218,218,218,218,218,218,218}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}",2011-04-05,T,None,"MULTIPOLYGON (((-88.00584 30.74350, -88.00625 30.74341, -88.00660 30.74339, ...",1950,0.285079
3,Alabama,3.0,73.0,87.0,001073087003,"{752,751,746,743,747,748,749,744,745,750}","{Russell,Macon,Dale,Barbour,Geneva,Henry,Houston,Bullock,Coffee,Lee}","{218,218,218,218,218,218,218,218,218,218}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}",2011-04-05,T,None,"POLYGON ((-85.47862 30.99720, -85.48830 30.99706, -85.49263 30.99700, -85.49...",1950,0.477668
4,Alabama,4.0,73.0,87.0,001073087004,"{760,757,759,758,755,753,756,754}","{Talladega,Dallas,""St. Clair"",Elmore,Clay,Autauga,Coosa,Calhoun}","{218,218,218,218,218,218,218,218}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}","{NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL}",2011-04-05,T,None,"POLYGON ((-87.01753 32.67771, -87.01760 32.66984, -87.01766 32.66327, -87.00...",1950,0.266304


In [ ]:
all_states_panel.head()

,PLACE_ID,NAME,year,n_districts,is_split,dist1,dist1_share,dist2,dist2_share,dist3,dist3_share,dist4,dist4_share,dist5,dist5_share,COUNTY,cook_year,dist1_pvi,dist1_pvi_party,dist1_pvi_string,dist2_pvi,dist2_pvi_party,dist2_pvi_string,dist3_pvi,dist3_pvi_party,dist3_pvi_string,dist4_pvi,dist4_pvi_party,dist4_pvi_string,dist5_pvi,dist5_pvi_party,dist5_pvi_string
0,5000911800,Canaan,1950,1,0,VT-01,0.99678,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5001127100,Franklin,1950,1,0,VT-01,0.998669,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5001105425,Berkshire,1950,1,0,VT-01,0.999732,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5001133025,Highgate,1950,1,0,VT-01,0.99754,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5001159125,Richford,1950,1,0,VT-01,0.999641,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
Govenors_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Political Data/us_governors_1775_2025.csv"
Presidents_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Political Data/Presidents - Sheet1 (1).csv"

gov = pd.read_csv(Govenors_path)
pres = pd.read_csv(Presidents_path)

      governor    state time_in_office       party  year
0  Mark Gordon  Wyoming    2019 - 2026  Republican  2019
1  Mark Gordon  Wyoming    2019 - 2026  Republican  2020
2  Mark Gordon  Wyoming    2019 - 2026  Republican  2021
3  Mark Gordon  Wyoming    2019 - 2026  Republican  2022
4  Mark Gordon  Wyoming    2019 - 2026  Republican  2023
   Year               Name Party
0  1789  George Washington   NaN
1  1790  George Washington   NaN
2  1791  George Washington   NaN
3  1792  George Washington   NaN
4  1793  George Washington   NaN


In [ ]:
# Loads in the governors data
gov = pd.read_csv(Govenors_path)
gov = gov[gov['year'].astype(str).str.strip().str.isdigit()]
gov['year'] = gov['year'].astype(int)
gov = gov[gov['year'] >= 1950].reset_index(drop=True)

# Parse term start year from "time_in_office" e.g. "2019 - 2023" → 2019
gov['term_start'] = gov['time_in_office'].str.extract(r'(\d{4})\s*-').astype(float).astype('Int64')

# loads in the presidential data
pres = pd.read_csv(Presidents_path)
pres = pres[pres['Year'] != 'ears (after inauguration)'].copy()
pres['Year'] = pres['Year'].astype(int)
pres = pres[pres['Year'] >= 1950].reset_index(drop=True)

In [ ]:
# Parse term start year from "time_in_office" e.g. "2019 - 2023" → 2019
gov['term_start'] = gov['time_in_office'].str.extract(r'(\d{4})\s*-').astype(float).astype('Int64')

# ── Dedup: on transition years, keep the INCOMING governor ───────────────────
def pick_governor(group):
    if len(group) == 1:
        return group
    incoming = group[group['term_start'] == group['year']]
    if len(incoming) == 1:
        return incoming
    return group.loc[[group['term_start'].idxmax()]]

gov_clean = (
    gov.groupby(['state', 'year'], group_keys=False)
       .apply(pick_governor)
       .reset_index(drop=True)
)

print(f"Before dedup: {len(gov)} rows")
print(f"After dedup:  {len(gov_clean)} rows")
remaining_dupes = (gov_clean.groupby(['state','year']).size() > 1).sum()
print(f"Remaining duplicate state+year pairs: {remaining_dupes}")

print(f"\nParty value counts:\n{gov_clean['party'].value_counts()}")
print(f"\nYear range: {gov_clean['year'].min()} - {gov_clean['year'].max()}")
print(f"Unique states: {gov_clean['state'].nunique()}")

print("\n=== Spot checks ===")
for state, year in [('Alaska', 1966), ('Alaska', 1970), ('California', 2011), ('Texas', 2000)]:
    before = gov[(gov['state']==state)&(gov['year']==year)][['governor','party','term_start']]
    after  = gov_clean[(gov_clean['state']==state)&(gov_clean['year']==year)][['governor','party','term_start']]
    print(f"\n{state} {year}:")
    print(f"  Before: {before.values.tolist()}")
    print(f"  After:  {after.values.tolist()}")













Before dedup: 4782 rows
After dedup:  4047 rows
Remaining duplicate state+year pairs: 0

Party value counts:
party
Democrat                               2035
Republican                             1873
Popular Democratic Party                 43
Independent                              37
New Progressive Party (Puerto Rico)      27
Independent Citizens Movement            13
Democratic-Farmer-Labor                  11
Federally-Appointed Civilian              8
Name: count, dtype: int64

Year range: 1950 - 2026
Unique states: 55

=== Spot checks ===

Alaska 1966:
  Before: [['Walter J. Hickel', 'Independent', 1966], ['William Allen Egan', 'Democrat', 1959]]
  After:  [['Walter J. Hickel', 'Independent', 1966]]

Alaska 1970:
  Before: [['William Allen Egan', 'Democrat', 1970], ['Keith H. Miller', 'Republican', 1969]]
  After:  [['William Allen Egan', 'Democrat', 1970]]

California 2011:
  Before: [['Edmund G. Brown Jr.', 'Democrat', 2011], ['Arnold Schwarzenegger', 'Republican', 2003]]